# 09 · Q5 : les niveaux de remise et la marge

**Projet 4 : Analyse des données de vente et retours** · Sprint 4

**Question de recherche Q5** : *quels niveaux de remise maximisent la marge ?*

**Objectif de l'énoncé servi par ce notebook** : évaluer l'efficacité des promotions et formuler des recommandations pour la rentabilité.

**Le cadre d'analyse** : comme dans `06_q1_facteurs_rentabilite`, les chiffres de **situation** et de **décision** (coût des remises, gains possibles) portent sur le dernier exercice, **2021**. Ils sont suivis sur les **trois exercices** : **2019**, l'année de référence où tous les magasins gagnaient de l'argent, **2020** et **2021**. Les **mécanismes** (ce que contient une ligne remisée, effet du niveau) sont mesurés sur les trois années réunies, puis vérifiés exercice par exercice. Les trois canaux sont étudiés séparément, avec l'accent sur les magasins.

| Étape | Ce qu'on analyse | Question posée |
|---|---|---|
| 1. Chargement | Ventes avec variables, table commune ventes et charges, charges, dimensions | |
| 2. Portrait des remises | Calendrier, niveaux annoncés et taux effectifs, par exercice et par canal | Que veut dire « une remise de 20 % » dans ces ventes ? |
| 3. Le coût des remises | Montant par canal et par exercice, face au résultat d'exploitation | Combien coûtent les remises, et qui les paie ? |
| 4. Magasin par magasin | Remise et ventes d'un magasin, à magasin et mois constants | Un magasin qui remise plus vend-il plus ? |
| 5. Avec et sans remise | Lignes remisées et non remisées du même produit, le même mois, par canal et par exercice | Une ligne remisée contient-elle plus d'unités ? Rapporte-t-elle plus de marge ? |
| 6. Le niveau | 7 %, 10 % et 20 % comparés produit par produit | Une remise plus forte rapporte-t-elle davantage ? |
| 7. Gamme et âge du produit | Remises selon la gamme, la nouveauté du produit et la catégorie, de 2019 à 2021 | Les remises suivent-elles la descente en gamme ? Une famille de produits fait-elle exception ? |
| 8. Les leviers sur l'exercice 2021 | Supprimer, plafonner ou cibler les remises, par canal, avec leur seuil ; le levier repris dans `06` | Que peut-on changer, et pour quel gain ? |
| 9. Le registre des tests | Correction de Bonferroni | Que peut-on conclure ? |
| 10. Synthèse | Réponse à Q5 | |

**Lit dans** : `data/clean/fait_ventes_variables.parquet`, `data/clean/fait_charges.parquet`, `data/clean/agregats/ventes_charges_boutique_mois.parquet`, `data/clean/dimensions`

**Écrit dans** : `reports/figures` (figures `09_*.png`) et `reports/livrables/L4_analyse/q5_tableaux.xlsx`

**Utilise** : `src/analyse.py` (fonctions communes aux notebooks d'analyse)

**Ce que ce notebook ne traite pas** : l'efficacité des promotions en tant que campagnes (nombre de ventes avant, pendant et après une promotion, calendrier des campagnes), étudiée dans `07_q2_promotions` ; les retours, étudiés dans `08_q3_q4_retours`. Ici on regarde **l'intérieur des lignes de vente** et **la marge en dollars**.

### Pourquoi on ne cherche pas un « taux optimal »

Q5 demande le taux de remise qui maximise la marge. Les données ne dessinent pas de courbe, pour deux raisons vérifiées en section 2 :

- il n'existe que **trois niveaux** dans les magasins européens, la boutique en ligne et le revendeur (7 %, 10 %, 20 %), plus un calendrier propre à un seul magasin (5 %, 10 %, 20 %) ;
- chaque niveau est **attaché à une période de l'année**, la même sur les trois exercices. Comparer deux niveaux, c'est aussi comparer deux saisons.

On répond donc à une question plus solide : **à produit égal et à mois égal, une ligne qui reçoit une remise contient-elle plus d'unités et rapporte-t-elle plus de marge qu'une ligne qui n'en reçoit pas, et cela dépend-il du niveau ?**

### Vocabulaire

| Terme | Définition |
|---|---|
| Ligne | le cumul d'un produit, dans une boutique, un jour donné, pour une promotion. Ce n'est pas un ticket de caisse |
| Taux annoncé | le taux de remise affiché par la promotion (5 %, 7 %, 10 % ou 20 %) |
| Unité remisée | unité de la ligne vendue avec la remise ; les autres unités de la ligne sont payées au plein tarif |
| Ligne remisée | ligne dont au moins une unité a reçu la remise |
| Taux de remise effectif | montant de la remise ÷ montant avant remise. Bien plus bas que le taux annoncé, puisque peu d'unités sont remisées |
| Marge en valeur | chiffre d'affaires net des retours moins coût des unités vendues et conservées, en dollars. C'est elle qu'on cherche à maximiser, pas le taux de marge |
| Couple (produit, mois) | un produit, un mois, un taux annoncé et un canal, pour lesquels on a à la fois des lignes remisées et des lignes non remisées |
| Nouveauté de l'année | produit vendu l'année de sa mise en vente (`dim_produit.date_mise_en_vente`), même définition que `06` et `06b` |
| Magasin ouvert toute l'année | magasin ouvert au 1er janvier de l'exercice. Seuls ces magasins sont comparés entre eux (section 4) |
| Intervalle de confiance à 95 % | la fourchette dans laquelle la vraie valeur se trouve très probablement, compte tenu du hasard de l'échantillon |

### Trois méthodes, et pourquoi

| Méthode | Ce qu'elle fait | Pourquoi ici |
|---|---|---|
| Centrage par magasin et par mois | retire de chaque observation la moyenne de son magasin et celle de son mois | pour comparer des magasins entre eux le même mois, donc **à saison égale** |
| Comparaison appariée | compare deux mesures prises sur la même unité (le même produit, le même mois, le même canal) | le produit explique 86 % du montant d'une ligne (`06_q1_facteurs_rentabilite`) : sans appariement, on comparerait des produits, pas des remises |
| Rééchantillonnage (bootstrap) | recalcule un résultat sur 1 000 échantillons tirés au hasard dans les données | pour donner une **fourchette** à chaque écart, et pas seulement un chiffre |

### Règle de lecture des tests

Chaque test est lu sur la valeur p **et** sur la taille de l'effet ; le seuil de 5 % est divisé par le nombre de tests (correction de Bonferroni, section 9). **Un test qui ne trouve rien ne prouve pas qu'il n'y a rien** : pour conclure qu'une ligne remisée ne contient pas plus d'unités, on s'appuie sur l'intervalle de confiance, qui dit jusqu'où l'écart peut aller.

## 0. Préparation

In [ ]:
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config  -> chemins, années de l'étude (ANNEES_ETUDE, ANNEE_REFERENCE, EXERCICE)
#   viz     -> charte graphique
#   analyse -> noter_test et tableau_tests (registre des tests), exporter_tableaux, libellés français
from src import config, viz, analyse
from src.analyse import LIBELLES_TYPES, ORDRE_TYPES, NOMS_MOIS, noter_test, tableau_tests

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---------------------------------------------------------------------------
# 3. Les trois exercices
# ---------------------------------------------------------------------------
ANNEES = config.ANNEES_ETUDE                  # [2019, 2020, 2021]
REFERENCE, EXERCICE = config.ANNEE_REFERENCE, config.EXERCICE
BASCULE = [a for a in ANNEES if a not in (REFERENCE, EXERCICE)][0]
COULEURS_ANNEES = {REFERENCE: viz.GRIS, BASCULE: viz.CATEGORIES[2], EXERCICE: viz.PRINCIPALE}
COULEURS_TYPES = {"En ligne": viz.CATEGORIES[2], "Revendeur": viz.ACCENT, "Magasins": viz.ALERTE}
print("Racine du projet :", config.ROOT)
print(f"Exercices : {ANNEES} · référence {REFERENCE} · exercice étudié {EXERCICE}")

# ---------------------------------------------------------------------------
# 4. Les niveaux de remise et les paramètres des tests
# ---------------------------------------------------------------------------
NIVEAUX = [0.07, 0.10, 0.20]                  # les trois niveaux du calendrier commun (le 5 % ne concerne qu'un magasin)
NOMS_NIVEAUX = {0.05: "5 %", 0.07: "7 %", 0.10: "10 %", 0.20: "20 %"}
COULEURS_NIVEAUX = {0.05: viz.SEQUENTIELLE[1], 0.07: viz.SEQUENTIELLE[2], 0.10: viz.SEQUENTIELLE[3], 0.20: viz.SEQUENTIELLE[5]}
ALPHA = 0.05                                  # seuil de signification avant correction
registre = []                                 # registre des tests d'hypothèse, rempli par noter_test()
ALEA = np.random.default_rng(2026)            # générateur aléatoire à graine fixe : mêmes résultats à chaque exécution
TIRAGES = 1000                                # nombre d'échantillons du rééchantillonnage


def m_dollars(valeur, decimales=1, signe=False):
    """Montant en millions de dollars, virgule française : 12,3 M$."""
    texte = f"{valeur / 1e6:{'+' if signe else ''},.{decimales}f}".replace(",", " ").replace(".", ",")
    return f"{texte} M$"


def pct(valeur, decimales=1, signe=True):
    """Taux en pourcentage, virgule française : 0.07 -> +7,0 %."""
    return f"{valeur:{'+' if signe else ''}.{decimales}%}".replace(".", ",").replace("%", " %")


# Tableaux à exporter vers Excel à la fin : nom de l'onglet -> tableau
tableaux = {}

## 1. Chargement

In [ ]:
chemins = {
    "ventes": config.CLEAN / "fait_ventes_variables.parquet",
    "charges": config.CLEAN / "fait_charges.parquet",
    "commun": config.AGREGATS / "ventes_charges_boutique_mois.parquet",
    "dim_boutique": config.CLEAN_DIMENSIONS / "dim_boutique.parquet",
    "dim_produit": config.CLEAN_DIMENSIONS / "dim_produit.parquet",
}
for nom, chemin in chemins.items():
    if not chemin.exists():
        raise FileNotFoundError(f"{chemin} introuvable : exécuter d'abord les notebooks 00 à 05c.")

ventes = pd.read_parquet(chemins["ventes"])
charges = pd.read_parquet(chemins["charges"])
commun = pd.read_parquet(chemins["commun"])
dim_boutique = pd.read_parquet(chemins["dim_boutique"])
dim_produit = pd.read_parquet(chemins["dim_produit"])

# ---------------------------------------------------------------------------
# 1. Variables de travail
# ---------------------------------------------------------------------------
ventes["type"] = ventes.type_boutique.map(LIBELLES_TYPES)
commun["type"] = commun.type_boutique.map(LIBELLES_TYPES)
reel = charges[charges.scenario == "Réel"].copy()
reel["type"] = reel.type_boutique.map(LIBELLES_TYPES)
ventes["unites_plein_tarif"] = ventes.quantite_vente - ventes.quantite_remise        # unités payées au plein tarif

# Magasin ouvert toute l'année : ouvert au 1er janvier de l'année de la vente
ouverture = dim_boutique.set_index("id_boutique").date_ouverture
ventes["annee_complete"] = ventes.id_boutique.map(ouverture) < pd.to_datetime(ventes.annee.astype(str) + "-01-02")

# Gamme et âge du produit (mêmes définitions que 06 et 06b)
LIBELLES_CLASSES = {"Economy": "Économique", "Regular": "Standard", "Deluxe": "Luxe"}
ORDRE_GAMMES = ["Économique", "Standard", "Luxe"]
ventes["gamme"] = ventes.classe.map(LIBELLES_CLASSES)
lancement = dim_produit.set_index("id_produit").date_mise_en_vente.dt.year
NOUVEAUTE, ANCIEN = "nouveauté de l'année", "produit plus ancien"
ventes["age"] = np.where(ventes.annee <= ventes.id_produit.map(lancement), NOUVEAUTE, ANCIEN)

# Lignes vendues pendant une promotion du calendrier commun (7 %, 10 % ou 20 %)
en_promotion = ventes[ventes.taux_remise_annonce.isin(NIVEAUX)].copy()

print(f"Ventes : {len(ventes):,} lignes · {ventes.id_boutique.nunique()} boutiques · {ventes.id_produit.nunique()} produits".replace(",", " "))
print(f"Période : du {ventes.date.min().date()} au {ventes.date.max().date()}")
print("Lignes en promotion à 7 %, 10 % ou 20 % :", f"{len(en_promotion):,}".replace(",", " "), f"soit {pct(len(en_promotion) / len(ventes), signe=False)} des lignes")
print("Lignes en promotion hors de la période de leur promotion (diagnostic 03, conservées) :",
      ventes[ventes.promo_hors_periode].groupby("annee").size().to_dict())

# Garde-fous : trois années complètes, et une table commune qui porte sur les mêmes ventes
if sorted(ventes.annee.unique()) != ANNEES or ventes.groupby("annee").trimestre.nunique().min() != 4:
    raise ValueError(f"La table des ventes ne couvre pas les années {ANNEES} en entier : relancer python -m src.extraction puis les notebooks 00 à 05c.")
if abs(commun.montant_remise.sum() - ventes.montant_remise.sum()) > 1 or abs(commun.ca_net.sum() - ventes.ca_net.sum()) > 1:
    raise ValueError("La table commune ventes et charges ne correspond pas à la table des ventes : relancer 05b_charges.")
# Un retour est remboursé au prix catalogue, remise ou non : supprimer une remise ne change donc pas le montant des retours
assert np.allclose(ventes.montant_retour, ventes.quantite_retour * ventes.prix_unitaire), "Retours : règle de remboursement inattendue"

## 2. Portrait des remises sur les trois exercices

Avant toute comparaison : quand chaque niveau s'applique-t-il, combien de lignes reçoivent réellement une remise, et que vaut cette remise une fois rapportée à tout ce qui est vendu ?

In [ ]:
# ---------------------------------------------------------------------------
# 1. Le calendrier : taux annoncé le plus fréquent, par mois et par exercice
# ---------------------------------------------------------------------------
calendrier = (ventes[ventes.nom_boutique != "Contoso Cambridge Store"]
              .groupby(["annee", "mois"]).taux_remise_annonce.agg(lambda s: s.value_counts().index[0]).unstack("mois"))
calendrier.columns = NOMS_MOIS
print("Taux annoncé le plus fréquent, par mois (toutes boutiques sauf Cambridge) :")
display(calendrier.style.format(lambda v: NOMS_NIVEAUX.get(round(v, 2), "aucun")))
print("Calendrier identique sur les trois exercices :", bool((calendrier.nunique() == 1).all()))
cambridge = (ventes[ventes.nom_boutique == "Contoso Cambridge Store"]
             .groupby(["annee", "mois"]).taux_remise_annonce.agg(lambda s: s.value_counts().index[0]).unstack("mois"))
cambridge.columns = NOMS_MOIS
print("Contoso Cambridge Store, seul magasin à suivre les promotions « North America » :")
display(cambridge.style.format(lambda v: NOMS_NIVEAUX.get(round(v, 2), "aucun")))
tableaux["2_calendrier"] = calendrier

# ---------------------------------------------------------------------------
# 2. Les niveaux : étendue, remise réelle, coût, par exercice
# ---------------------------------------------------------------------------
def portrait(lignes, cles):
    """Une ligne par groupe : volume, part des lignes remisées, unités remisées, taux effectif, montant des remises."""
    g = lignes.groupby(cles, observed=True)
    t = g.agg(lignes=("id_vente", "size"), part_lignes_remisees=("a_remise", "mean"),
              unites=("quantite_vente", "sum"), unites_remisees=("quantite_remise", "sum"),
              montant_avant_remise=("montant_avant_remise", "sum"), montant_remise=("montant_remise", "sum"),
              ca_net=("ca_net", "sum"), marge_nette=("marge_nette", "sum"))
    t["unites_par_ligne"] = t.unites / t.lignes
    t["unites_remisees_par_ligne_remisee"] = lignes[lignes.a_remise].groupby(cles, observed=True).quantite_remise.mean()
    t["part_unites_remisees"] = t.unites_remisees / t.unites
    t["taux_remise_effectif"] = t.montant_remise / t.montant_avant_remise
    t["remises_par_ligne"] = t.montant_remise / t.lignes
    return t[["lignes", "part_lignes_remisees", "unites_par_ligne", "unites_remisees_par_ligne_remisee", "part_unites_remisees",
              "taux_remise_effectif", "montant_remise", "remises_par_ligne", "ca_net"]]


FORMATS_PORTRAIT = {"lignes": "{:,.0f}", "part_lignes_remisees": "{:.1%}", "unites_par_ligne": "{:.1f}",
                    "unites_remisees_par_ligne_remisee": "{:.2f}", "part_unites_remisees": "{:.1%}",
                    "taux_remise_effectif": "{:.2%}", "montant_remise": "{:,.0f}", "remises_par_ligne": "{:,.1f}", "ca_net": "{:,.0f}"}
niveaux_annee = portrait(ventes, ["taux_remise_annonce", "annee"])
print("Les niveaux de remise, par exercice (0 % : périodes sans promotion) :")
display(niveaux_annee.style.format(FORMATS_PORTRAIT, na_rep="").format_index(lambda v: NOMS_NIVEAUX.get(round(v, 2), "0 %") if isinstance(v, float) else v, level=0))
tableaux["2_niveaux_par_exercice"] = niveaux_annee

# La remise d'une ligne : taux annoncé × prix × unités remisées. Vérification sur les lignes remisées.
remisees = ventes[ventes.a_remise]
rapport = remisees.montant_remise / (remisees.taux_remise_annonce * remisees.prix_unitaire * remisees.quantite_remise)
print(f"Remise d'une ligne ÷ (taux annoncé × prix × unités remisées) : médiane {rapport.median():.3f}, ".replace(".", ",") +
      f"{pct((rapport.sub(1).abs() < 0.001).mean(), signe=False)} des lignes remisées à exactement 1")
repartition = (remisees.quantite_remise.clip(upper=5).groupby(remisees.annee).value_counts(normalize=True).unstack().rename(columns={5: "5 ou plus"}))
print("Nombre d'unités remisées dans une ligne remisée (part des lignes remisées), par exercice :")
display(repartition.style.format("{:.1%}"))
tableaux["2_unites_remisees"] = repartition

# ---------------------------------------------------------------------------
# 3. Par canal et par exercice
# ---------------------------------------------------------------------------
canal = portrait(ventes, ["type", "annee"]).loc[ORDRE_TYPES]
canal_promo = portrait(ventes[ventes.taux_remise_annonce > 0], ["type", "annee"]).loc[ORDRE_TYPES]
canal["part des lignes remisées (en promotion)"] = canal_promo.part_lignes_remisees
canal["remises ÷ CA net"] = canal.montant_remise / canal.ca_net
canal = canal.drop(columns="part_lignes_remisees")
print("Par canal et par exercice (toutes les lignes ; la part des lignes remisées est calculée sur les périodes de promotion) :")
display(canal.style.format(FORMATS_PORTRAIT | {"part des lignes remisées (en promotion)": "{:.1%}", "remises ÷ CA net": "{:.2%}"}))
tableaux["2_canaux"] = canal

COLONNES_EVOL = ["montant_remise", "ca_net", "unites_par_ligne", "unites_remisees_par_ligne_remisee", "part_unites_remisees", "taux_remise_effectif"]
evol = pd.concat({f"{a} → {b}": (canal.xs(b, level="annee") / canal.xs(a, level="annee") - 1)[COLONNES_EVOL].stack()
                  for a, b in [(REFERENCE, BASCULE), (BASCULE, EXERCICE), (REFERENCE, EXERCICE)]}, axis=1)
evol = evol.reindex(pd.MultiIndex.from_product([ORDRE_TYPES, COLONNES_EVOL]))
print("Évolutions par canal :")
display(evol.style.format("{:+.1%}"))
tableaux["2_canaux_evolution"] = evol

# ---------------------------------------------------------------------------
# 4. Graphique : montant des remises par canal et par exercice, selon le niveau
# ---------------------------------------------------------------------------
cout_niveau = ventes[ventes.taux_remise_annonce > 0].pivot_table(index=["type", "annee"], columns="taux_remise_annonce",
                                                                 values="montant_remise", aggfunc="sum").fillna(0).loc[ORDRE_TYPES]
fig, ax = plt.subplots(figsize=(10.5, 4.6))
positions = [i * 4 + j for i in range(len(ORDRE_TYPES)) for j in range(len(ANNEES))]
bas = np.zeros(len(cout_niveau))
for taux in cout_niveau.columns:
    ax.bar(positions, cout_niveau[taux], bottom=bas, color=COULEURS_NIVEAUX[round(taux, 2)], width=0.8, label=f"Remise à {NOMS_NIVEAUX[round(taux, 2)]}")
    bas += cout_niveau[taux].values
for x, total in zip(positions, bas):
    ax.text(x, total + 0.15e6, f"{total / 1e6:.1f}".replace(".", ","), ha="center", fontsize=9, color=viz.FONCE)
ax.set_xticks(positions, [str(a) for _ in ORDRE_TYPES for a in ANNEES], fontsize=9)
for i, t in enumerate(ORDRE_TYPES):
    ax.text(i * 4 + 1, -0.12, t, transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=10.5, fontweight="bold", color=viz.FONCE)
viz.format_dollars(ax, axe="y")
ax.legend(frameon=False, loc="upper center", ncol=2)
ax.set_ylim(0, bas.max() * 1.3)
viz.titre(ax, "Les remises des magasins ont fondu de plus de moitié, celles des autres canaux beaucoup moins",
          "Montant des remises accordées, en millions de dollars, par canal, par exercice et par niveau de remise")
print("Figure enregistrée :", viz.enregistrer(fig, "09_cout_des_remises"))
plt.show()

**Lecture**

**Le calendrier est le même sur les trois exercices.** 20 % en janvier et d'octobre à décembre, 7 % de février à avril, rien de mai à juillet, 10 % en août et septembre. Un seul magasin, Contoso Cambridge, suit un autre calendrier (5 % de janvier à mars, 10 % de juillet à septembre, 20 % en novembre et décembre), lui aussi identique chaque année. Aucun mois n'a connu deux niveaux différents d'un exercice à l'autre : **on ne peut pas séparer le niveau de la saison**.

**Une remise porte toujours sur très peu d'unités.** La remise d'une ligne vaut exactement taux annoncé × prix × unités remisées (99,4 % des lignes remisées). Une ligne remisée contient 1 à 4 unités remisées dans 91 % à 92 % des cas selon l'exercice, 2,9 en moyenne, quel que soit le niveau et l'exercice.

| Niveau | Part des lignes remisées (2019 · 2020 · 2021) | Taux effectif (2019 · 2020 · 2021) | Remises 2021 |
|---|---|---|---|
| 7 % (févr. à avr.) | 53,6 % · 57,0 % · 54,6 % | 0,85 % · 0,81 % · 0,85 % | 1,5 M$ |
| 10 % (août, sept.) | 81,3 % · 81,6 % · 81,1 % | 2,44 % · 2,22 % · 2,10 % | 2,6 M$ |
| 20 % (oct. à janv.) | 82,2 % · 82,3 % · 82,0 % | 5,46 % · 4,91 % · 4,70 % | 9,8 M$ |

Le niveau à 20 % fait à lui seul 70 % à 72 % du coût des remises chaque exercice (15,8, 10,9 puis 9,8 M$) : il combine le taux le plus élevé, la plus grande étendue et la période la plus longue (4 mois sur 12).

**Par canal, la même mécanique, des évolutions différentes.**

| | Remises 2019 → 2021 | Évolution | CA net 2019 → 2021 | Taux effectif 2019 · 2020 · 2021 | Unités par ligne 2019 → 2021 |
|---|---|---|---|---|---|
| En ligne | 5,6 → 5,3 M$ | −6,3 % | +6,7 % | 2,15 % · 1,94 % · 1,90 % | 12,7 → 25,8 |
| Revendeur | 4,0 → 3,1 M$ | −22,0 % | −14,1 % | 2,14 % · 1,94 % · 1,95 % | 12,7 → 22,3 |
| Magasins | 12,4 → 5,5 M$ | −55,8 % | −48,6 % | 2,33 % · 2,03 % · 2,01 % | 9,4 → 17,1 |

- La part des lignes remisées pendant les promotions ne bouge pas (71 % en magasin, 74 % à 77 % ailleurs).
- Le nombre d'unités remisées dans une ligne remisée non plus (2,5 en magasin, 3,0 à 3,4 ailleurs, −1,7 % à −9,1 % en deux exercices), alors que les **lignes ont doublé de taille** (+81 % à +103 % d'unités par ligne). La part des unités remisées est donc divisée par deux (magasins 14,5 % → 7,7 %).
- Le taux effectif recule beaucoup moins (−9 % à −14 % en deux exercices) : la section 7 montre que ce recul vient presque entièrement de la gamme économique, où les lignes ont le plus grossi.
- Les remises des magasins baissent plus vite que leurs ventes (−55,8 % contre −48,6 %), parce qu'ils ont perdu des lignes (`06`) et que chaque ligne garde le même nombre d'unités remisées.

## 3. Le coût des remises face au résultat

On rapporte le montant des remises au chiffre d'affaires et au **résultat d'exploitation** de chaque canal (`05b_charges`). C'est ce qui dit si les remises sont un sujet pour la rentabilité.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Remises, chiffre d'affaires et résultat d'exploitation, par canal et par exercice
# ---------------------------------------------------------------------------
cout = commun.groupby(["type", "annee"])[["ca_net", "montant_avant_remise", "montant_remise", "marge_nette", "resultat_exploitation"]].sum().loc[ORDRE_TYPES]
ensemble = commun.groupby("annee")[["ca_net", "montant_avant_remise", "montant_remise", "marge_nette", "resultat_exploitation"]].sum()
ensemble.index = pd.MultiIndex.from_product([["Ensemble"], ensemble.index], names=["type", "annee"])
cout = pd.concat([cout, ensemble])
cout["remises ÷ CA net"] = cout.montant_remise / cout.ca_net
cout["remises ÷ marge nette"] = cout.montant_remise / cout.marge_nette
cout["résultat sans remise (à ventes inchangées)"] = cout.resultat_exploitation + cout.montant_remise
cout["part des remises de l'entreprise"] = cout.montant_remise / cout.montant_remise.xs("Ensemble", level="type").reindex(cout.index.get_level_values("annee")).values
print("Remises et résultat d'exploitation, par canal et par exercice :")
display(cout.style.format({"ca_net": "{:,.0f}", "montant_avant_remise": "{:,.0f}", "montant_remise": "{:,.0f}", "marge_nette": "{:,.0f}",
                           "resultat_exploitation": "{:,.0f}", "remises ÷ CA net": "{:.2%}", "remises ÷ marge nette": "{:.2%}",
                           "résultat sans remise (à ventes inchangées)": "{:,.0f}", "part des remises de l'entreprise": "{:.1%}"}))
tableaux["3_cout_des_remises"] = cout

# Remises par magasin, magasins ouverts toute l'année
remises_magasin = (ventes[(ventes.type == "Magasins") & ventes.annee_complete].groupby(["annee", "id_boutique"]).montant_remise.sum()
                   .groupby("annee").describe()[["count", "min", "50%", "max"]])
print("Remises accordées par un magasin ouvert toute l'année :")
display(remises_magasin.style.format({"count": "{:.0f}", "min": "{:,.0f}", "50%": "{:,.0f}", "max": "{:,.0f}"}))
tableaux["3_remises_par_magasin"] = remises_magasin

**Lecture**

**Les remises coûtent environ 2 % du chiffre d'affaires, chaque exercice et dans chaque canal.**

| Remises | 2019 | 2020 | 2021 |
|---|---|---|---|
| Ensemble | 22,1 M$ · 2,33 % du CA | 15,5 M$ · 2,05 % | 13,9 M$ · 2,02 % |
| Magasins | 12,4 M$ · 2,43 % | 6,4 M$ · 2,10 % | 5,5 M$ · 2,08 % |
| Part des magasins dans les remises | 56,3 % | 41,5 % | 39,5 % |
| Résultat d'exploitation des magasins | 121,6 M$ | 13,0 M$ | −10,8 M$ |
| Résultat des magasins sans remise, à ventes inchangées | 134,0 M$ | 19,4 M$ | −5,3 M$ |

- **Les remises ne font pas le déficit des magasins**, mais elles en font la moitié : sans elles, à ventes inchangées, le déficit de 2021 passerait de 10,8 à 5,3 M$ (5,4 M$ une fois déduites les « autres charges » qui suivent les ventes, section 8). En 2019, elles pesaient 10 % du résultat des magasins ; en 2021, 5,5 M$ face à un résultat négatif.
- En ligne (5,3 M$) et chez le revendeur (3,1 M$), elles pèsent 5 % et 6 % d'un résultat largement positif.
- Un magasin ouvert toute l'année accorde 102 000 $ de remises en 2021 (médiane), contre 254 000 $ en 2019. Le plus remiseur est Moscow No.2 (176 000 $), le magasin qui vend le plus (`06`) : les remises suivent les ventes du magasin, il n'y a pas de magasin qui remise à part.

## 4. Magasin par magasin : un magasin qui remise plus vend-il plus ?

**La question.** Un mois donné, tous les magasins sont sous la même promotion, mais ils n'accordent pas exactement la même remise : la part des lignes remisées varie un peu d'un magasin à l'autre. Si la remise fait vendre, les magasins qui remisent plus que d'habitude doivent vendre plus que d'habitude.

**Le piège.** Le niveau de remise suit le calendrier, et les magasins n'ont pas la même taille. Une corrélation calculée directement mesurerait la saison et la taille, pas la remise.

**La méthode.** On retire de chaque observation (un magasin, un mois) la moyenne de son magasin sur l'exercice et la moyenne de son mois. Ce qui reste est l'écart d'un magasin, un mois donné, par rapport à ce qu'on attendait de lui. On travaille sur les magasins ouverts toute l'année et sur les mois de promotion à 7 %, 10 % ou 20 %.

**La mesure de la remise.** On retient la **part des lignes remisées**. Le taux effectif est aussi montré, mais il baisse mécaniquement quand les lignes contiennent plus d'unités (une ligne remisée contient toujours 1 à 4 unités remisées, section 2) : il serait corrélé négativement aux unités sans que la remise y soit pour rien.

- **H0** : à magasin et mois constants, aucun lien entre la part des lignes remisées et les ventes (unités, marge).
- **Test** : corrélation de Spearman sur les écarts centrés, les trois exercices réunis, puis exercice par exercice.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Table magasin × mois, mois de promotion, magasins ouverts toute l'année
# ---------------------------------------------------------------------------
mag_promo = en_promotion[(en_promotion.type == "Magasins") & en_promotion.annee_complete]
bm = (mag_promo.groupby(["id_boutique", "annee", "annee_mois"])
      .agg(lignes=("id_vente", "size"), unites=("quantite_vente", "sum"), ca_net=("ca_net", "sum"),
           marge_nette=("marge_nette", "sum"), montant_remise=("montant_remise", "sum"),
           montant_avant_remise=("montant_avant_remise", "sum"), part_lignes_remisees=("a_remise", "mean")).reset_index())
bm["taux_remise_effectif"] = bm.montant_remise / bm.montant_avant_remise
bm["magasin_exercice"] = bm.id_boutique.astype(str) + "_" + bm.annee.astype(str)
CIBLES = {"lignes": "Nombre de lignes", "unites": "Unités vendues", "ca_net": "Chiffre d'affaires net", "marge_nette": "Marge en valeur"}
MESURES = {"part_lignes_remisees": "Part des lignes remisées", "taux_remise_effectif": "Taux de remise effectif"}
for cible in CIBLES:
    bm[cible] = np.log(bm[cible])          # logarithme : on raisonne en variation relative
print(f"Table magasin × mois de promotion : {len(bm):,} observations · {bm.id_boutique.nunique()} magasins · {bm.annee_mois.nunique()} mois".replace(",", " "))


def centrer(table, colonne, passes=20):
    """Retire d'une colonne la moyenne de son magasin sur l'exercice et la moyenne de son mois.

    Il reste l'écart d'un magasin, un mois donné, par rapport à ce qu'on attend de ce magasin et de ce mois.
    Les deux moyennes sont retirées tour à tour, plusieurs fois, parce que tous les magasins ne sont pas
    présents tous les mois (tableau incomplet).
    """
    x = table[colonne] - table[colonne].mean()
    for _ in range(passes):
        x = x - x.groupby(table.magasin_exercice).transform("mean")
        x = x - x.groupby(table.annee_mois).transform("mean")
    return x


# ---------------------------------------------------------------------------
# 2. Corrélations brutes, puis à magasin et mois constants, trois exercices réunis et par exercice
# ---------------------------------------------------------------------------
lignes_tableau = []
for periode, table in [("2019 à 2021", bm)] + [(str(a), bm[bm.annee == a].reset_index(drop=True)) for a in ANNEES]:
    for mesure, nom_mesure in MESURES.items():
        for cible, nom_cible in CIBLES.items():
            brute, _ = stats.spearmanr(table[mesure], table[cible])
            nette, p_nette = stats.spearmanr(centrer(table, mesure), centrer(table, cible))
            lignes_tableau.append({"période": periode, "mesure de la remise": nom_mesure, "variable": nom_cible, "n": len(table),
                                   "corrélation brute": brute, "à magasin et mois constants": nette, "p": p_nette})
correlations = pd.DataFrame(lignes_tableau)
vue = correlations.pivot_table(index=["mesure de la remise", "variable"], columns="période", values="à magasin et mois constants", sort=False)
print("Corrélation de Spearman à magasin et mois constants :")
display(vue.style.format("{:+.2f}"))
brutes = correlations[correlations.période == "2019 à 2021"].set_index(["mesure de la remise", "variable"])[["corrélation brute", "à magasin et mois constants", "p"]]
print("Trois exercices réunis, avant et après centrage :")
display(brutes.style.format({"corrélation brute": "{:+.2f}", "à magasin et mois constants": "{:+.2f}", "p": "{:.2g}"}))
tableaux["4_correlations"] = correlations

reste = centrer(bm, "part_lignes_remisees")
print(f"Part des lignes remisées : écart-type brut {pct(bm.part_lignes_remisees.std(), signe=False)}, à magasin et mois constants {pct(reste.std(), signe=False)}")

for cible in ["unites", "marge_nette"]:
    ligne = brutes.loc[("Part des lignes remisées", CIBLES[cible])]
    noter_test(registre, f"Magasins : à magasin et mois constants, remiser plus de lignes va-t-il avec plus de {CIBLES[cible].lower()} ?",
               "Aucun lien entre la part des lignes remisées et la variable", "Spearman sur écarts centrés", len(bm),
               f"ρ = {ligne['à magasin et mois constants']:+.2f}".replace(".", ","), ligne["p"])

**Lecture**

**Un magasin qui remise plus de lignes que d'habitude ne vend pas plus que d'habitude.**

| Part des lignes remisées et… | Corrélation brute | À magasin et mois constants (2019 à 2021) | Par exercice (2019 · 2020 · 2021) |
|---|---|---|---|
| Nombre de lignes | +0,05 | −0,01 (p = 0,73) | −0,13 · +0,04 · +0,02 |
| Unités vendues | −0,08 | −0,03 (p = 0,30) | −0,03 · −0,04 · −0,01 |
| Chiffre d'affaires net | −0,09 | −0,04 (p = 0,20) | −0,05 · −0,05 · −0,02 |
| Marge en valeur | −0,10 | −0,04 (p = 0,12) | −0,04 · −0,04 · −0,04 |

- Sur 1 356 observations (53 magasins, mois de promotion des trois exercices), aucune corrélation ne dépasse 0,13 en valeur absolue, et aucune n'est positive avec la marge. Les deux tests du registre conservent H0.
- Le taux effectif donne des corrélations un peu plus négatives (−0,08 et −0,09 avec les unités et la marge ; −0,21 à −0,24 en brut), mais c'est la mécanique décrite plus haut : un mois où les lignes sont plus grosses, le même nombre d'unités remisées pèse moins.
- **Limite** : une fois le magasin et le mois retirés, la part des lignes remisées ne varie plus que de 7,6 % d'écart-type (25,7 % en brut). Le test peut manquer un effet modéré. C'est pourquoi la section suivante change d'échelle et regarde les lignes elles-mêmes.

**Ce qu'on en retient** : au niveau d'un magasin, rien n'indique que la remise fasse vendre davantage. Les corrélations de `06` (taux de remise effectif −0,29 avec le chiffre d'affaires d'un magasin) mesurent surtout le calendrier et la taille des lignes.

## 5. Avec et sans remise, à produit et mois égaux

**L'idée.** Pendant une promotion, toutes les lignes ne reçoivent pas de remise (de 54 % à 82 % selon le niveau, section 2). Pour un même produit, un même mois et un même canal, on compare les lignes remisées aux lignes non remisées : même produit, même saison, même promotion, seule la remise change.

**Deux questions.**

1. **De quoi est faite une ligne remisée ?** Les unités remisées s'ajoutent-elles à ce qui aurait été acheté, ou remplacent-elles des unités qui auraient été payées au plein tarif ? Dans le premier cas la remise crée des ventes, dans le second elle ne fait que baisser le prix.
2. **Que rapporte-t-elle ?** La marge en valeur d'une ligne remisée est-elle plus haute ou plus basse que celle d'une ligne non remisée ?

**Deux façons de résumer, à lire ensemble.** L'écart **des moyennes** répond à « combien au total » ; le **couple médian** donne le cas typique. Quand les deux divergent, c'est qu'une minorité de lignes fait la moyenne. Toutes les lignes sont gardées, y compris les grandes quantités repérées au nettoyage (28 % des lignes de 2021, parce que les quantités par ligne ont doublé, `06`) ; la section se termine par une vérification sans elles.

**Le constat repris dans `06`** : « en magasin, une ligne remisée ne contient pas plus d'unités qu'une autre ». On le vérifie ici **sur chaque exercice et chaque canal**. On le tient pour vérifié quand l'intervalle de confiance du supplément d'unités reste sous +2 %.

- **H0 (unités)** : pour un même produit et un même mois, une ligne remisée et une ligne non remisée contiennent le même nombre d'unités. Test de Wilcoxon apparié sur le logarithme du rapport, par canal et par exercice.
- **H0 (marge)** : elles ont la même marge. Test de Wilcoxon apparié sur l'écart de marge, par canal, les trois exercices réunis.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Les couples (produit, mois) : lignes remisées et non remisées, dans un même canal
# ---------------------------------------------------------------------------
def couples_produit_mois(lignes, cles_en_plus=()):
    """Compare, pour chaque couple (taux annoncé, produit, mois), les lignes remisées et les lignes non remisées.

    Renvoie une ligne par couple qui a les deux sortes de lignes, avec la moyenne par ligne de la marge, du chiffre
    d'affaires net, des unités, des unités au plein tarif et des unités remisées, suffixées _avec et _sans.
    cles_en_plus : colonnes ajoutées à la clé (par exemple la boutique, pour comparer dans le même magasin).
    """
    cles = ["taux_remise_annonce", *cles_en_plus, "id_produit", "annee_mois"]
    g = (lignes.groupby(cles + ["a_remise"], observed=True)
         .agg(marge=("marge_nette", "mean"), ca=("ca_net", "mean"), unites=("quantite_vente", "mean"),
              plein_tarif=("unites_plein_tarif", "mean"), remisees=("quantite_remise", "mean"), lignes=("id_vente", "size")))
    large = g.unstack("a_remise").dropna()
    large.columns = [f"{mesure}_{'avec' if remise else 'sans'}" for mesure, remise in large.columns]
    large = large.reset_index()
    large["annee"] = large.annee_mois.str[:4].astype(int)
    large["age"] = np.where(large.annee <= large.id_produit.map(lancement), NOUVEAUTE, ANCIEN)
    large["gamme"] = large.id_produit.map(dim_produit.set_index("id_produit").classe.map(LIBELLES_CLASSES))
    large["categorie"] = large.id_produit.map(dim_produit.set_index("id_produit").categorie)
    return large


def ecart_moyennes(avec, sans, tirages=TIRAGES):
    """Écart relatif des moyennes (moyenne avec ÷ moyenne sans − 1), et son intervalle de confiance à 95 %.

    L'intervalle est obtenu par rééchantillonnage : on tire au hasard, avec remise, autant de couples qu'il y en a,
    on recalcule l'écart, et on recommence `tirages` fois ; on écarte les 2,5 % de résultats les plus bas et les plus hauts.
    """
    avec, sans = np.asarray(avec, dtype=float), np.asarray(sans, dtype=float)
    valeur = avec.mean() / sans.mean() - 1
    simules = np.empty(tirages)
    for k in range(tirages):
        tirage = ALEA.integers(0, len(avec), len(avec))
        simules[k] = avec[tirage].mean() / sans[tirage].mean() - 1
    bas, haut = np.percentile(simules, [2.5, 97.5])
    return valeur, bas, haut


def resumer_couples(c):
    """Résumé d'un ensemble de couples : composition des lignes, chiffre d'affaires et marge par ligne."""
    unites, u_bas, u_haut = ecart_moyennes(c.unites_avec, c.unites_sans)
    gain, g_bas, g_haut = ecart_moyennes(c.marge_avec, c.marge_sans)
    positifs = c.marge_sans > 0
    return {"couples (produit, mois)": len(c),
            "unités, ligne sans remise": c.unites_sans.mean(), "unités, ligne avec remise": c.unites_avec.mean(),
            "dont au plein tarif": c.plein_tarif_avec.mean(), "dont remisées": c.remisees_avec.mean(),
            "unités en plus (moyennes)": unites, "unités en plus, borne basse": u_bas, "unités en plus, borne haute": u_haut,
            "unités en plus (couple médian)": np.exp(np.median(np.log(c.unites_avec / c.unites_sans))) - 1,
            "part des unités remisées qui remplacent": (c.unites_sans.mean() - c.plein_tarif_avec.mean()) / c.remisees_avec.mean(),
            "CA par ligne, sans remise": c.ca_sans.mean(), "CA par ligne, avec remise": c.ca_avec.mean(),
            "écart de CA par ligne": c.ca_avec.mean() / c.ca_sans.mean() - 1,
            "marge par ligne, sans remise": c.marge_sans.mean(), "marge par ligne, avec remise": c.marge_avec.mean(),
            "gain de marge (moyennes)": gain, "gain de marge, borne basse": g_bas, "gain de marge, borne haute": g_haut,
            "gain de marge (couple médian)": (c.marge_avec / c.marge_sans - 1)[positifs].median(),
            "couples où la ligne remisée rapporte plus": (c.marge_avec > c.marge_sans).mean()}


FORMATS_COUPLES = {"couples (produit, mois)": "{:,.0f}", "unités, ligne sans remise": "{:.2f}", "unités, ligne avec remise": "{:.2f}",
                   "dont au plein tarif": "{:.2f}", "dont remisées": "{:.2f}",
                   "unités en plus (moyennes)": "{:+.1%}", "unités en plus, borne basse": "{:+.1%}", "unités en plus, borne haute": "{:+.1%}",
                   "unités en plus (couple médian)": "{:+.1%}", "part des unités remisées qui remplacent": "{:.0%}",
                   "CA par ligne, sans remise": "{:,.0f}", "CA par ligne, avec remise": "{:,.0f}", "écart de CA par ligne": "{:+.1%}",
                   "marge par ligne, sans remise": "{:,.0f}", "marge par ligne, avec remise": "{:,.0f}",
                   "gain de marge (moyennes)": "{:+.1%}", "gain de marge, borne basse": "{:+.1%}", "gain de marge, borne haute": "{:+.1%}",
                   "gain de marge (couple médian)": "{:+.1%}", "couples où la ligne remisée rapporte plus": "{:.0%}"}

couples_type = {t: couples_produit_mois(en_promotion[en_promotion.type == t]) for t in ORDRE_TYPES}

# ---------------------------------------------------------------------------
# 2. Par canal, les trois exercices réunis, puis par canal et par exercice
# ---------------------------------------------------------------------------
par_type = pd.DataFrame({t: resumer_couples(c) for t, c in couples_type.items()}).T
par_type_annee = pd.DataFrame({(t, a): resumer_couples(c[c.annee == a]) for t, c in couples_type.items() for a in ANNEES}).T
par_type_annee.index.names = ["type", "annee"]
COLONNES_COMPOSITION = ["couples (produit, mois)", "unités, ligne sans remise", "unités, ligne avec remise", "dont au plein tarif", "dont remisées",
                        "unités en plus (moyennes)", "unités en plus (couple médian)", "part des unités remisées qui remplacent",
                        "écart de CA par ligne", "gain de marge (moyennes)", "gain de marge, borne basse", "gain de marge, borne haute",
                        "gain de marge (couple médian)", "couples où la ligne remisée rapporte plus"]
print("Lignes remisées et non remisées du même produit, le même mois, dans le même canal · trois exercices réunis :")
display(par_type[COLONNES_COMPOSITION].style.format(FORMATS_COUPLES))
print("Les mêmes mesures, exercice par exercice :")
display(par_type_annee[COLONNES_COMPOSITION].style.format(FORMATS_COUPLES))
tableaux["5_avec_sans_par_canal"] = par_type
tableaux["5_avec_sans_par_exercice"] = par_type_annee

# ---------------------------------------------------------------------------
# 3. Le constat repris dans 06 : une ligne remisée contient-elle plus d'unités ? Chaque canal, chaque exercice
# ---------------------------------------------------------------------------
SEUIL_UNITES = 0.02          # au-delà de +2 % d'unités dans l'intervalle, on ne peut plus dire « pas plus d'unités »
verification = par_type_annee[["couples (produit, mois)", "unités en plus (moyennes)", "unités en plus, borne basse", "unités en plus, borne haute",
                               "unités en plus (couple médian)", "part des unités remisées qui remplacent"]].copy()
for (t, a) in verification.index:
    c = couples_type[t][couples_type[t].annee == a]
    p = stats.wilcoxon(np.log(c.unites_avec / c.unites_sans)).pvalue
    verification.loc[(t, a), "p"] = p
    noter_test(registre, f"{t}, {a} : une ligne remisée contient-elle plus d'unités qu'une ligne non remisée du même produit, le même mois ?",
               "Même nombre d'unités avec et sans remise", "Wilcoxon apparié", len(c),
               f"écart des moyennes = {pct(verification.loc[(t, a), 'unités en plus (moyennes)'])}", p)
verification["pas plus d'unités ?"] = np.where(verification["unités en plus, borne haute"] < SEUIL_UNITES,
                                               "oui : vérifié", "non : la ligne remisée contient plus d'unités")
print("Une ligne remisée contient-elle plus d'unités qu'une autre ? (borne haute de l'intervalle sous +2 % : vérifié)")
display(verification.style.format(FORMATS_COUPLES | {"p": "{:.2g}"}))
tableaux["5_verification_constat_06"] = verification

# Vérification de robustesse : dans le même magasin, et sans les lignes de grande quantité
mag = en_promotion[en_promotion.type == "Magasins"]
robustesse = {}
for a in ANNEES:
    meme_magasin = couples_produit_mois(mag[mag.annee == a], cles_en_plus=["id_boutique"])
    sans_grandes = couples_produit_mois(mag[(mag.annee == a) & ~mag.grande_quantite])
    for nom, c in [("même produit, même mois, même magasin", meme_magasin), ("sans les lignes de grande quantité", sans_grandes)]:
        u, bas, haut = ecart_moyennes(c.unites_avec, c.unites_sans)
        robustesse[(nom, a)] = {"couples": len(c), "unités en plus (moyennes)": u, "borne basse": bas, "borne haute": haut,
                                "part des unités remisées qui remplacent": (c.unites_sans.mean() - c.plein_tarif_avec.mean()) / c.remisees_avec.mean()}
robustesse = pd.DataFrame(robustesse).T
print("Magasins : la même vérification, dans le même magasin, puis sans les lignes de grande quantité")
display(robustesse.style.format({"couples": "{:,.0f}", "unités en plus (moyennes)": "{:+.1%}", "borne basse": "{:+.1%}", "borne haute": "{:+.1%}",
                                 "part des unités remisées qui remplacent": "{:.0%}"}))
tableaux["5_robustesse_magasins"] = robustesse

hors_magasins = {}
for t in ["En ligne", "Revendeur"]:
    lignes_t = en_promotion[(en_promotion.type == t) & ~en_promotion.grande_quantite]
    for a in ANNEES:
        c = couples_produit_mois(lignes_t[lignes_t.annee == a])
        hors_magasins[(t, a)] = {"couples": len(c), "unités en plus (moyennes)": c.unites_avec.mean() / c.unites_sans.mean() - 1,
                                 "unités en plus (couple médian)": np.exp(np.median(np.log(c.unites_avec / c.unites_sans))) - 1,
                                 "gain de marge (moyennes)": c.marge_avec.mean() / c.marge_sans.mean() - 1}
hors_magasins = pd.DataFrame(hors_magasins).T
print("En ligne et revendeur, sans les lignes de grande quantité :")
display(hors_magasins.style.format({"couples": "{:,.0f}", "unités en plus (moyennes)": "{:+.1%}", "unités en plus (couple médian)": "{:+.1%}",
                                    "gain de marge (moyennes)": "{:+.1%}"}))
tableaux["5_robustesse_hors_magasins"] = hors_magasins

# Taille des lignes : les lignes remisées sont-elles plus souvent de grandes lignes ?
taille = (en_promotion.groupby(["type", "annee", "a_remise"]).agg(lignes=("id_vente", "size"), unites_par_ligne=("quantite_vente", "mean"),
                                                                    part_grandes_quantites=("grande_quantite", "mean"))
          .rename(index={False: "sans remise", True: "avec remise"}, level="a_remise").reindex(ORDRE_TYPES, level="type"))
print("Taille des lignes en promotion, selon le canal, l'exercice et la présence d'une remise :")
display(taille.unstack("a_remise").style.format({c: ("{:,.0f}" if c[0] == "lignes" else "{:.1f}" if c[0] == "unites_par_ligne" else "{:.1%}") for c in taille.unstack("a_remise").columns}))
tableaux["5_taille_des_lignes"] = taille

# ---------------------------------------------------------------------------
# 4. La marge, par canal, trois exercices réunis
# ---------------------------------------------------------------------------
for t in ORDRE_TYPES:
    c = couples_type[t]
    r = par_type.loc[t]
    noter_test(registre, f"{t} : une ligne remisée rapporte-t-elle plus de marge qu'une ligne non remisée du même produit, le même mois ?",
               "Même marge par ligne avec et sans remise", "Wilcoxon apparié", len(c),
               f"écart des moyennes = {pct(r['gain de marge (moyennes)'])}, couple médian = {pct(r['gain de marge (couple médian)'])}",
               stats.wilcoxon(c.marge_avec - c.marge_sans).pvalue)

# ---------------------------------------------------------------------------
# 5. Figure : de quoi est faite une ligne, avec et sans remise, par canal et par exercice
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.8), sharey=True)
for ax, t in zip(axes, ORDRE_TYPES):
    for rang, a in enumerate(ANNEES):
        r = par_type_annee.loc[(t, a)]
        x_sans, x_avec = rang * 3, rang * 3 + 1
        ax.bar(x_sans, r["unités, ligne sans remise"], color=viz.GRIS, width=0.85, label="Ligne sans remise" if rang == 0 else None)
        ax.bar(x_avec, r["dont au plein tarif"], color=viz.PRINCIPALE, width=0.85, label="Ligne avec remise : unités au plein tarif" if rang == 0 else None)
        ax.bar(x_avec, r["dont remisées"], bottom=r["dont au plein tarif"], color=viz.ACCENT, width=0.85, label="Ligne avec remise : unités remisées" if rang == 0 else None)
        for x, h in ((x_sans, r["unités, ligne sans remise"]), (x_avec, r["unités, ligne avec remise"])):
            ax.text(x, h + 0.3, f"{h:.1f}".replace(".", ","), ha="center", fontsize=8.5, color=viz.FONCE)
    ax.set_xticks([rang * 3 + 0.5 for rang in range(len(ANNEES))], [str(a) for a in ANNEES])
    ax.set_title(t, fontsize=12, pad=6, color=viz.FONCE)
axes[0].set_ylabel("Unités par ligne")
axes[0].set_ylim(0, par_type_annee["unités, ligne avec remise"].max() * 1.3)
axes[0].legend(frameon=False, loc="upper left", fontsize=8.5)
fig.suptitle("En magasin, une ligne remisée contient autant d'unités qu'une autre, chaque exercice",
             x=0.01, ha="left", fontsize=14, fontweight="bold", color=viz.FONCE)
fig.text(0.01, 0.905, "Composition moyenne d'une ligne en promotion, même produit, même mois, même canal · toutes les lignes",
         fontsize=10, color=viz.GRIS)
fig.subplots_adjust(top=0.8, wspace=0.06)
print("Figure enregistrée :", viz.enregistrer(fig, "09_unites_avec_sans_remise"))
plt.show()

**Lecture**

**En magasin : une ligne remisée contient autant d'unités qu'une autre, chaque exercice. Le constat repris dans `06` est vérifié.**

| Magasins | 2019 | 2020 | 2021 | 2019 à 2021 |
|---|---|---|---|---|
| Couples (produit, mois) | 11 134 | 9 078 | 6 610 | 26 822 |
| Unités, ligne sans remise · avec remise | 8,98 · 9,04 | 12,19 · 12,01 | 17,05 · 16,73 | 12,06 · 11,94 |
| Unités en plus (intervalle à 95 %) | +0,7 % (+0,4 à +1,0) | −1,5 % (−2,0 à −1,0) | −1,9 % (−2,7 à −0,9) | −1,0 % |
| Part des unités remisées qui remplacent du plein tarif | 98 % | 107 % | 113 % | 105 % |
| Marge d'une ligne remisée | −7,1 % | −8,4 % | −8,1 % | −7,8 % (−8,1 à −7,6) |

- Les 2,5 unités remisées d'une ligne **remplacent une pour une** des unités qui auraient été payées au plein tarif. La remise ne fait pas acheter plus, elle fait payer moins cher : le chiffre d'affaires d'une ligne remisée est plus bas de 4,8 %, sa marge de 7,8 % (couple médian −7,7 %), et la ligne remisée ne rapporte plus que dans 15 % des couples.
- Le test de Wilcoxon rejette H0 chaque exercice, mais dans les deux sens (+0,7 %, puis −1,5 % et −1,9 %) : avec des milliers de couples, des écarts de 1 à 2 % sont « significatifs ». C'est l'intervalle qui compte : sa borne haute ne dépasse pas +1,0 % (2019) et reste négative en 2020 et 2021.
- **Deux vérifications** confirment l'ordre de grandeur : dans le **même magasin** (même produit, même mois), +2,1 %, +1,3 % puis +0,6 % d'unités, avec 93 % à 97 % de substitution ; **sans les grandes quantités**, +0,3 %, −1,3 % et −0,7 %. Seule la comparaison dans le même magasin en 2019 dépasse le seuil de +2 % retenu ici (+2,1 %, intervalle jusqu'à +2,9 %) ; même au plus haut de cette fourchette, l'écart reste sous la baisse de 5,2 % qui annulerait le gain d'une suppression des remises (section 8).

**En ligne et chez le revendeur : les lignes remisées sont nettement plus grosses.**

| | En ligne 2019 · 2020 · 2021 | Revendeur 2019 · 2020 · 2021 |
|---|---|---|
| Unités en plus (moyennes) | +44,7 % · +37,0 % · +31,9 % | +33,8 % · +27,2 % · +18,9 % |
| Unités en plus (couple médian) | +50,0 % · +33,3 % · +20,0 % | +33,3 % · 0,0 % · 0,0 % |
| Marge en plus (moyennes) | +35,0 % · +26,6 % · +27,9 % | +25,2 % · +16,8 % · +15,4 % |
| Couples où la ligne remisée rapporte plus | 68 % · 61 % · 55 % | 57 % · 45 % · 36 % |

- Le constat de `06` **ne vaut pas** pour ces deux canaux : la ligne remisée contient plus d'unités chaque exercice, et l'écart se réduit d'année en année.
- **Les unités au plein tarif sont elles aussi plus nombreuses** dans une ligne remisée (en ligne, 14,6 contre 13,1 sans remise, trois exercices réunis). Les unités remisées ne remplacent donc rien : c'est la ligne entière qui est plus grosse.
- Chez le revendeur, le cas typique ne bouge pas (couple médian à 0 % en 2020 et 2021, marge médiane −1,1 %) : l'écart moyen vient d'une minorité de grosses lignes. En ligne, le cas typique est lui aussi plus gros (+20 % en 2021).
- Sans les lignes de grande quantité, l'écart moyen tombe à +12,0 % en ligne et +8,4 % chez le revendeur en 2021, avec un couple médian à 0 % : une bonne part de l'écart tient aux très grosses lignes.

*Interprétation métier (hypothèse)* : **en magasin**, le client achète ce qu'il était venu acheter, et la remise en caisse sur quelques unités lui fait seulement payer moins. **En ligne et chez le revendeur**, deux lectures restent possibles, que les données ne départagent pas : soit la remise pousse une partie des acheteurs à commander plus (ce qui serait logique pour un professionnel qui groupe ses achats), soit les grosses commandes obtiennent plus facilement une remise, sans que celle-ci les ait provoquées. Dans le premier cas la remise crée de la marge, dans le second elle en coûte. La section 8 chiffre les deux.

## 6. Le niveau de remise change-t-il quelque chose ?

On compare les trois niveaux **dans chaque canal** : pour chaque niveau, ce que la ligne remisée contient d'unités en plus, ce qu'elle fait de chiffre d'affaires et de marge en plus. Puis on vérifie **produit par produit** si le niveau change le gain de marge, sur les produits présents aux trois niveaux, avec des lignes remisées et non remisées à chacun.

- **H0** : pour un même produit, le gain de marge d'une ligne remisée est le même à 7 %, à 10 % et à 20 %.
- **Tests** : Friedman (trois mesures sur les mêmes produits, comparées par leurs rangs, sans être tiré par quelques produits très chers), puis Wilcoxon apparié entre les deux niveaux extrêmes.

**Limite à garder en tête** : chaque niveau correspond à une période de l'année, la même sur les trois exercices. La comparaison est faite à produit égal, pas à saison égale.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Canal × niveau, trois exercices réunis, puis canal × niveau × exercice
# ---------------------------------------------------------------------------
croise = pd.DataFrame({(t, NOMS_NIVEAUX[n]): resumer_couples(c[c.taux_remise_annonce == n]) for t, c in couples_type.items() for n in NIVEAUX}).T
croise.index.names = ["type", "niveau"]
COLONNES_NIVEAU = ["couples (produit, mois)", "unités en plus (moyennes)", "unités en plus, borne basse", "unités en plus, borne haute",
                   "unités en plus (couple médian)", "écart de CA par ligne", "gain de marge (moyennes)", "gain de marge, borne basse",
                   "gain de marge, borne haute", "gain de marge (couple médian)"]
print("Par canal et par niveau, trois exercices réunis :")
display(croise[COLONNES_NIVEAU].style.format(FORMATS_COUPLES))
tableaux["6_canal_et_niveau"] = croise

niveau_annee = pd.DataFrame({(t, NOMS_NIVEAUX[n], a): {"unités en plus": c.loc[sel, "unites_avec"].mean() / c.loc[sel, "unites_sans"].mean() - 1,
                                                        "gain de marge": c.loc[sel, "marge_avec"].mean() / c.loc[sel, "marge_sans"].mean() - 1}
                             for t, c in couples_type.items() for n in NIVEAUX for a in ANNEES
                             for sel in [(c.taux_remise_annonce == n) & (c.annee == a)]}).T
niveau_annee.index.names = ["type", "niveau", "annee"]
niveau_annee = niveau_annee.reindex(pd.MultiIndex.from_product([ORDRE_TYPES, [NOMS_NIVEAUX[n] for n in NIVEAUX], ANNEES], names=niveau_annee.index.names))
print("Par canal, par niveau et par exercice (écart des moyennes) :")
display(niveau_annee.unstack("annee").style.format("{:+.1%}"))
tableaux["6_niveau_par_exercice"] = niveau_annee

# ---------------------------------------------------------------------------
# 2. Produit par produit : le niveau change-t-il le gain de marge ?
# ---------------------------------------------------------------------------
lignes_niveaux = []
for t, c in couples_type.items():
    par_produit = c.groupby(["taux_remise_annonce", "id_produit"])[["marge_avec", "marge_sans", "unites_avec", "unites_sans", "ca_avec", "ca_sans"]].mean()
    gains = (par_produit.marge_avec / par_produit.marge_sans - 1).unstack("taux_remise_annonce").dropna()
    unites_plus = (par_produit.unites_avec / par_produit.unites_sans - 1).unstack("taux_remise_annonce").reindex(gains.index)
    _, p_friedman = stats.friedmanchisquare(*[gains[n] for n in NIVEAUX])
    medianes = ", ".join(f"{NOMS_NIVEAUX[n]} : {pct(gains[n].median())}" for n in NIVEAUX)
    noter_test(registre, f"Niveaux, {t.lower()} : le gain de marge d'une ligne remisée dépend-il du niveau de remise ?",
               "Même gain de marge aux trois niveaux", "Friedman", len(gains), f"gain médian par produit, {medianes}", p_friedman)
    _, p_extremes = stats.wilcoxon(gains[0.20], gains[0.07])
    ecart = (gains[0.20] - gains[0.07]).median() * 100
    noter_test(registre, f"Niveaux, {t.lower()} : une remise à 20 % rapporte-t-elle plus de marge par ligne qu'une remise à 7 % ?",
               "Même gain de marge à 7 % et à 20 %", "Wilcoxon apparié", len(gains),
               f"écart médian (20 % moins 7 %) = {ecart:+.1f} points".replace(".", ","), p_extremes)
    for n in NIVEAUX:
        lignes_niveaux.append({"type": t, "niveau": NOMS_NIVEAUX[n], "produits": len(gains),
                               "gain de marge, produit médian": gains[n].median(),
                               "unités en plus, produit médian": unites_plus[n].median(),
                               "produits où la ligne remisée rapporte plus": (gains[n] > 0).mean(),
                               "p Friedman": p_friedman, "p 20 % contre 7 %": p_extremes})
par_niveau = pd.DataFrame(lignes_niveaux).set_index(["type", "niveau"])
print("Produit par produit, sur les produits présents aux trois niveaux dans le canal :")
display(par_niveau.style.format({"produits": "{:,.0f}", "gain de marge, produit médian": "{:+.1%}", "unités en plus, produit médian": "{:+.1%}",
                                 "produits où la ligne remisée rapporte plus": "{:.0%}", "p Friedman": "{:.2g}", "p 20 % contre 7 %": "{:.2g}"}))
tableaux["6_par_niveau_produits"] = par_niveau

# ---------------------------------------------------------------------------
# 3. Figure : unités en plus et gain de marge par niveau, pour chaque canal
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.9))
for ax, (mesure, bas, haut, titre_panneau) in zip(axes, [
        ("unités en plus (moyennes)", "unités en plus, borne basse", "unités en plus, borne haute", "Unités en plus dans une ligne remisée"),
        ("gain de marge (moyennes)", "gain de marge, borne basse", "gain de marge, borne haute", "Marge en plus d'une ligne remisée")]):
    for rang_type, t in enumerate(ORDRE_TYPES):
        xs = np.arange(len(NIVEAUX)) + (rang_type - 1) * 0.24
        valeurs = croise.loc[t]
        for x, (_, r) in zip(xs, valeurs.iterrows()):
            ax.plot([x, x], [r[bas], r[haut]], color=COULEURS_TYPES[t], linewidth=6, alpha=0.4, solid_capstyle="butt")
            ax.text(x + 0.05, r[mesure], pct(r[mesure], 0), va="center", fontsize=8, color=viz.FONCE)
        ax.scatter(xs, valeurs[mesure], s=70, color=COULEURS_TYPES[t], zorder=3, label=t)
    ax.axhline(0, color=viz.GRIS, linewidth=1)
    ax.set_xticks(range(len(NIVEAUX)), [f"Remise à {NOMS_NIVEAUX[n]}" for n in NIVEAUX])
    ax.set_xlim(-0.5, len(NIVEAUX) - 0.35)
    viz.format_pourcent(ax, axe="y")
    ax.set_title(titre_panneau, fontsize=11.5, pad=6, color=viz.FONCE)
axes[1].legend(frameon=False, loc="upper right")
fig.suptitle("Le niveau à 20 % ne fait nulle part le plus de marge ; en magasin, plus la remise est forte, plus la ligne perd",
             x=0.01, ha="left", fontsize=14, fontweight="bold", color=viz.FONCE)
fig.text(0.01, 0.905, "Écart des moyennes entre lignes remisées et non remisées du même produit, le même mois · trait épais : intervalle à 95 % · 2019 à 2021",
         fontsize=10, color=viz.GRIS)
fig.subplots_adjust(top=0.8, wspace=0.18)
print("Figure enregistrée :", viz.enregistrer(fig, "09_gain_de_marge_par_niveau"))
plt.show()

**Lecture**

**En magasin : plus la remise est forte, plus la ligne perd de marge, chaque exercice.**

| Magasins | 7 % | 10 % | 20 % |
|---|---|---|---|
| Unités en plus (2019 à 2021) | −2,3 % | −1,0 % | +0,1 % |
| Chiffre d'affaires d'une ligne | −3,3 % | −3,7 % | −6,4 % |
| Marge d'une ligne, 2019 · 2020 · 2021 | −3,9 % · −5,5 % · −3,5 % | −4,4 % · −6,3 % · −6,8 % | −10,7 % · −11,6 % · −11,5 % |
| Produit médian (1 980 produits) | −3,6 % | −6,1 % | −11,5 % |
| Produits où la ligne remisée rapporte plus | 19 % | 10 % | 5 % |

Le niveau ne change rien aux unités (aucun intervalle ne s'écarte de zéro de plus de 3 %) : il ne change que le prix cédé. La différence entre niveaux est massive (Friedman, p = 2,3 × 10⁻²¹⁹ ; 20 % contre 7 % : −7,4 points pour le produit médian).

**En ligne et chez le revendeur : le niveau à 20 % est celui qui rapporte le moins.**

| Marge d'une ligne remisée (2019 à 2021) | 7 % | 10 % | 20 % |
|---|---|---|---|
| En ligne · unités en plus | +33,8 % | +40,5 % | +36,2 % |
| En ligne · marge en plus | +30,6 % | +39,1 % | +23,6 % |
| Revendeur · unités en plus | +28,8 % | +23,4 % | +25,8 % |
| Revendeur · marge en plus | +24,7 % | +21,5 % | +14,7 % |

- **Les unités en plus ne grandissent pas avec le niveau** : une remise à 20 % ne va pas avec de plus grosses lignes qu'une remise à 7 % ou 10 %, alors qu'elle coûte deux à trois fois plus cher par unité remisée.
- La marge en plus est donc la plus faible à 20 %, dans les deux canaux et **chacun des trois exercices** (en ligne : +29,1 %, +22,9 %, +19,9 % ; revendeur : +21,5 %, +11,0 %, +11,0 %).
- Produit par produit, l'écart est confirmé : −3,6 points entre 20 % et 7 % en ligne (p = 1,1 × 10⁻⁵), −9,0 points chez le revendeur (p = 5,2 × 10⁻¹¹). En ligne, le meilleur niveau est 10 % (+32,7 % pour le produit médian) ; chez le revendeur, 7 % (+16,1 %).

**Ce qu'on en retient pour Q5**

1. **Nulle part le niveau à 20 % ne rapporte plus de marge** qu'un niveau plus bas, à produit égal.
2. En magasin, le niveau qui maximise la marge est **l'absence de remise**.
3. En ligne et chez le revendeur, si la remise sert, un niveau de 7 % à 10 % fait au moins aussi bien que 20 %.

**Réserve** : chaque niveau tombe à une saison (20 % d'octobre à janvier). Ces écarts sont mesurés à produit égal, pas à saison égale : on ne peut pas exclure que les lignes remisées de l'automne soient différentes pour une autre raison que le taux.

## 7. Gamme et âge du produit

`06b_q1_chute_des_ventes` établit deux mouvements de fond : la clientèle **descend en gamme** (la gamme économique passe de 10,6 % à 21,6 % du chiffre d'affaires) et l'assortiment **vieillit** (552, 429 puis 272 lancements). Trois questions :

1. Les remises portent-elles autant sur toutes les gammes, sur les nouveautés et sur les produits anciens ?
2. Comment ont-elles évolué de 2019 à 2021 face à la descente en gamme : le recul du taux effectif vient-il du changement de gamme ou d'autre chose ?
3. Une gamme, un âge ou une catégorie de produit fait-il exception au constat de la section 5 ?

In [ ]:
# ---------------------------------------------------------------------------
# 1. Remises par gamme et par âge du produit, sur les trois exercices
# ---------------------------------------------------------------------------
def portrait_segment(lignes, colonne, ordre):
    """Portrait des remises par exercice et par segment, avec les parts dans le chiffre d'affaires et dans les remises."""
    t = portrait(lignes, ["annee", colonne])
    t["part des lignes remisées (en promotion)"] = portrait(lignes[lignes.taux_remise_annonce > 0], ["annee", colonne]).part_lignes_remisees
    t["part du CA"] = t.ca_net / t.groupby(level="annee").ca_net.transform("sum")
    t["part des remises"] = t.montant_remise / t.groupby(level="annee").montant_remise.transform("sum")
    t["remises ÷ CA net"] = t.montant_remise / t.ca_net
    colonnes = ["part du CA", "part des remises", "montant_remise", "remises ÷ CA net", "taux_remise_effectif",
                "part des lignes remisées (en promotion)", "unites_par_ligne", "unites_remisees_par_ligne_remisee", "part_unites_remisees"]
    return t[colonnes].reindex(ordre, level=colonne)


FORMATS_SEGMENT = FORMATS_PORTRAIT | {"part du CA": "{:.1%}", "part des remises": "{:.1%}", "remises ÷ CA net": "{:.2%}",
                                      "part des lignes remisées (en promotion)": "{:.1%}"}
par_gamme = portrait_segment(ventes, "gamme", ORDRE_GAMMES)
print("Remises par gamme et par exercice, toutes boutiques :")
display(par_gamme.style.format(FORMATS_SEGMENT))
par_age = portrait_segment(ventes, "age", [NOUVEAUTE, ANCIEN])
print("Remises selon l'âge du produit, par exercice, toutes boutiques :")
display(par_age.style.format(FORMATS_SEGMENT))
tableaux["7_remises_par_gamme"] = par_gamme
tableaux["7_remises_par_age"] = par_age

taux_canal_gamme = ventes.groupby(["type", "gamme", "annee"])[["montant_remise", "montant_avant_remise"]].sum()
taux_canal_gamme = (taux_canal_gamme.montant_remise / taux_canal_gamme.montant_avant_remise).unstack("annee").reindex(
    pd.MultiIndex.from_product([ORDRE_TYPES, ORDRE_GAMMES]))
print("Taux de remise effectif par canal, par gamme et par exercice :")
display(taux_canal_gamme.style.format("{:.2%}"))
tableaux["7_taux_effectif_canal_gamme"] = taux_canal_gamme

# ---------------------------------------------------------------------------
# 2. Le recul du taux effectif : changement de gamme (ou d'âge), ou recul à l'intérieur de chaque segment ?
# ---------------------------------------------------------------------------
# taux effectif = somme sur les segments de (part du montant avant remise × taux effectif du segment)
#   effet de structure : ce qu'aurait changé le seul déplacement des ventes entre segments, à taux de l'année de départ
#   effet de taux      : ce qu'a changé le recul du taux à l'intérieur de chaque segment, à la structure de l'année d'arrivée
def decomposer(lignes, colonne, depart, arrivee):
    t = lignes.groupby(["annee", colonne])[["montant_avant_remise", "montant_remise"]].sum()
    part = t.montant_avant_remise / t.groupby(level="annee").montant_avant_remise.transform("sum")
    taux = t.montant_remise / t.montant_avant_remise
    total = t.groupby(level="annee").montant_remise.sum() / t.groupby(level="annee").montant_avant_remise.sum()
    structure = ((part.xs(arrivee) - part.xs(depart)) * taux.xs(depart)).sum()
    effet_taux = (part.xs(arrivee) * (taux.xs(arrivee) - taux.xs(depart))).sum()
    return {f"taux effectif au départ": total[depart], f"taux effectif à l'arrivée": total[arrivee], "variation (points)": total[arrivee] - total[depart],
            "effet de structure (points)": structure, "effet de taux (points)": effet_taux,
            "remises de l'arrivée aux taux du départ": (t.xs(arrivee).montant_avant_remise * taux.xs(depart)).sum(),
            "remises de l'arrivée": t.xs(arrivee).montant_remise.sum()}


decomposition = pd.DataFrame({(perimetre, nom_segment, f"{a} → {b}"): decomposer(lignes, colonne, a, b)
                              for perimetre, lignes in [("Toutes boutiques", ventes), ("Magasins", ventes[ventes.type == "Magasins"])]
                              for nom_segment, colonne in [("gamme", "gamme"), ("âge du produit", "age")]
                              for a, b in [(REFERENCE, BASCULE), (BASCULE, EXERCICE), (REFERENCE, EXERCICE)]}).T
decomposition.index.names = ["périmètre", "segment", "période"]
print("Le taux de remise effectif, décomposé en effet de structure et effet de taux :")
display(decomposition.style.format({c: ("{:,.0f}" if c.startswith("remises") else "{:+.2%}" if "points" in c else "{:.2%}") for c in decomposition.columns}))
tableaux["7_decomposition_taux_effectif"] = decomposition

# ---------------------------------------------------------------------------
# 3. Une gamme, un âge ou une catégorie fait-il exception ? (couples des trois exercices réunis)
# ---------------------------------------------------------------------------
GROUPES = {"Magasins": ["Magasins"], "En ligne et revendeur": ["En ligne", "Revendeur"]}
lignes_segment = []
for groupe, types in GROUPES.items():
    c_groupe = pd.concat([couples_type[t] for t in types])
    for nom_segment, colonne, valeurs in [("Gamme", "gamme", ORDRE_GAMMES), ("Âge du produit", "age", [NOUVEAUTE, ANCIEN]),
                                          ("Catégorie", "categorie", sorted(c_groupe.categorie.unique()))]:
        for valeur in valeurs:
            c = c_groupe[c_groupe[colonne] == valeur]
            if len(c) < 30:                    # trop peu de couples pour une estimation
                continue
            u, u_bas, u_haut = ecart_moyennes(c.unites_avec, c.unites_sans)
            g, g_bas, g_haut = ecart_moyennes(c.marge_avec, c.marge_sans)
            lignes_segment.append({"boutiques": groupe, "segment": nom_segment, "valeur": valeur, "couples": len(c),
                                   "unités en plus": u, "unités, borne basse": u_bas, "unités, borne haute": u_haut,
                                   "gain de marge": g, "marge, borne basse": g_bas, "marge, borne haute": g_haut,
                                   "gain de marge (couple médian)": (c.marge_avec / c.marge_sans - 1)[c.marge_sans > 0].median()})
segments = pd.DataFrame(lignes_segment)
print("Lignes remisées et non remisées du même produit, le même mois, par segment de produit (2019 à 2021) :")
display(segments.style.format({"couples": "{:,.0f}"} | {c: "{:+.1%}" for c in segments.columns if c not in ("boutiques", "segment", "valeur", "couples")}).hide(axis="index"))
tableaux["7_par_segment"] = segments

# ---------------------------------------------------------------------------
# 4. Figure : gain de marge par segment, un panneau par groupe de boutiques
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 6.2), sharey=True)
COULEURS_SEGMENTS = {"Gamme": viz.ACCENT, "Âge du produit": viz.CATEGORIES[2], "Catégorie": viz.PRINCIPALE}
ordre_valeurs = segments[segments.boutiques == "Magasins"].valeur.tolist()[::-1]
for ax, groupe in zip(axes, GROUPES):
    s_groupe = segments[segments.boutiques == groupe].set_index("valeur").reindex(ordre_valeurs)
    for position, (valeur, s) in enumerate(s_groupe.iterrows()):
        if pd.isna(s["gain de marge"]):
            continue
        couleur = COULEURS_SEGMENTS[s.segment]
        ax.plot([s["marge, borne basse"], s["marge, borne haute"]], [position, position], color=couleur, linewidth=5, alpha=0.45, solid_capstyle="butt")
        ax.scatter(s["gain de marge"], position, s=60, color=couleur, zorder=3)
    ax.axvline(0, color=viz.GRIS, linewidth=1)
    ax.set_yticks(range(len(ordre_valeurs)), ordre_valeurs)
    viz.format_pourcent(ax, axe="x")
    ax.set_title(groupe, fontsize=12, pad=6, color=viz.FONCE)
for nom_segment, couleur in COULEURS_SEGMENTS.items():
    axes[1].scatter([], [], s=60, color=couleur, label=nom_segment)
axes[1].legend(frameon=False, loc="center left")
fig.suptitle("En magasin, la remise fait perdre de la marge dans toutes les gammes, tous les âges et toutes les catégories",
             x=0.01, ha="left", fontsize=14, fontweight="bold", color=viz.FONCE)
fig.text(0.01, 0.92, "Marge en plus d'une ligne remisée, avec son intervalle à 95 % · même produit, même mois, même canal · 2019 à 2021",
         fontsize=10, color=viz.GRIS)
fig.subplots_adjust(top=0.85, wspace=0.06, left=0.2)
print("Figure enregistrée :", viz.enregistrer(fig, "09_gain_par_segment"))
plt.show()

**Lecture**

**1. Les remises ne distinguent ni les gammes, ni les nouveautés.** La part des lignes remisées pendant les promotions est la même partout (73 % à 75 % dans les trois gammes, 73 % à 75 % pour les nouveautés comme pour les produits anciens), et une ligne remisée contient toujours 2,9 unités remisées. Les nouveautés reçoivent une part des remises proche de leur part du chiffre d'affaires (12,9 % contre 11,8 % en 2021). Aucune politique de remise par produit n'est visible : la remise s'applique au catalogue entier.

**2. Les remises n'ont pas suivi la descente en gamme : elles ont reculé sur la gamme économique.**

| Gamme économique | 2019 | 2020 | 2021 |
|---|---|---|---|
| Part du chiffre d'affaires | 10,6 % | 14,2 % | 21,6 % |
| Part des remises | 18,0 % | 13,1 % | 10,4 % |
| Taux de remise effectif | 3,71 % | 1,84 % | 0,96 % |
| Unités par ligne | 8,7 | 19,1 | 39,4 |
| Part des unités remisées | 18,0 % | 8,4 % | 4,0 % |

Le taux effectif des gammes standard (1,97 % → 2,16 %) et luxe (2,33 % → 2,34 %) est stable. Celui de la gamme économique est divisé par quatre, dans les trois canaux (de 3,5 % à 3,9 % en 2019, à 0,9 % à 1,0 % en 2021) : ses lignes sont devenues **quatre fois et demie plus grosses**, et elles gardent le même nombre d'unités remisées.

| Taux effectif, toutes boutiques | 2019 → 2020 | 2020 → 2021 | 2019 → 2021 |
|---|---|---|---|
| Variation | −0,27 point | −0,03 point | −0,30 point (2,25 % → 1,95 %) |
| dont déplacement des ventes entre gammes | +0,05 | −0,01 | +0,18 |
| dont recul du taux à l'intérieur des gammes | −0,32 | −0,02 | −0,48 |

- À elle seule, la descente en gamme aurait **fait monter** le taux effectif (+0,18 point), puisque la gamme économique était la plus remisée en 2019. C'est le recul du taux à l'intérieur de la gamme économique qui l'a fait baisser.
- Le recul s'est fait presque entièrement en 2020 (−0,27 point sur −0,30). En magasin, même lecture (+0,17 et −0,49 point).
- Aux taux de 2019 par gamme, les remises de 2021 auraient coûté 17,3 M$ au lieu de 13,9 M$ (magasins : 6,8 au lieu de 5,5 M$).
- L'âge du produit ne joue pas : le vieillissement de l'assortiment n'explique rien du recul (effet de structure +0,01 point).

*Interprétation métier (hypothèse)* : la remise semble plafonnée à quelques unités par ligne. Quand un client achète en grande quantité un produit d'entrée de gamme, la remise ne suit pas sa commande. La baisse du taux effectif n'est donc pas une décision de politique commerciale : elle tient à la taille des lignes.

**3. Aucune famille de produits ne fait exception.**

- **Magasins** : dans les trois gammes, les deux âges et les huit catégories, la ligne remisée perd de la marge, et aucun intervalle n'atteint zéro. La perte va de −1,6 % (jeux et jouets) à −9,8 % (électroménager, la première catégorie en chiffre d'affaires), −9,2 % pour la gamme économique, −7,9 % pour les nouveautés. Les unités en plus restent entre −1,6 % et +0,4 %.
- **En ligne et revendeur** : partout la ligne remisée est plus grosse (+24,6 % à +35,4 % d'unités) et rapporte plus (+17,6 % pour les jeux et jouets à +30,2 % pour la musique et les films), sans écart net entre gammes (+24,2 % à +26,0 %) ni entre nouveautés et produits anciens (+26,9 % et +24,7 %).

**Conséquence** : il n'existe pas de famille de produits pour laquelle une remise en magasin se justifierait par la marge, et pas de famille, en ligne et chez le revendeur, où elle serait plus « efficace » qu'ailleurs. Cibler les remises par produit ne s'appuie sur aucun écart mesurable : le ciblage pertinent est **par canal**.

## 8. Les leviers sur l'exercice 2021

On passe du constat à l'action. Chaque levier est chiffré sur le **dernier exercice**, canal par canal. Trois familles :

| Levier | Ce qui change | Lignes concernées |
|---|---|---|
| **Supprimer** | plus aucune remise dans le canal | toutes les lignes remisées |
| **Plafonner** | le niveau à 20 % ramené à 10 %, ou les niveaux à 10 % et 20 % ramenés à 7 % | lignes des promotions à 10 % et 20 % |
| **Cibler** | plus de remise sur les nouveautés de l'année, remises gardées sur les produits plus anciens | lignes remisées des nouveautés |

**Deux lectures pour encadrer chaque gain.** On ne sait pas si les unités en plus d'une ligne remisée (en ligne et chez le revendeur) **viennent** de la remise, ou si la remise va aux lignes qui sont de toute façon plus grosses. Chaque levier est donc chiffré deux fois :

| Lecture | Hypothèse | Gain de résultat |
|---|---|---|
| **A** | Les lignes resteraient les mêmes : la remise ne fait rien acheter de plus | Le montant des remises récupérées, moins les « autres charges » qui suivent les ventes (1,1 % du chiffre d'affaires, `06`) |
| **B** | Sans remise, une ligne remisée ressemblerait à une ligne non remisée du même produit, le même mois | L'écart de marge mesuré en sections 5 à 7 (trois exercices réunis), appliqué à la marge des lignes remisées de 2021 |

**Quand les deux lectures donnent le même signe, la décision est sûre.** Quand elles s'opposent, il faut un essai avant de décider.

**Le seuil.** Les deux lectures portent sur l'intérieur des lignes. Reste un risque qu'aucune ne mesure : que des clients viennent moins souvent sans remise (moins de lignes), question qui relève de l'efficacité des promotions comme campagnes (`07_q2_promotions`). Pour chaque levier, on donne la **baisse des ventes qui annulerait le gain** de la lecture A :

> seuil = gain ÷ (ventes exposées × ce que rapporte 1 $ de vente)

avec 0,551 $ par dollar de vente en magasin et 0,552 $ ailleurs et pour l'ensemble (`06`, recalculés ci-dessous de la même façon), et pour ventes exposées le chiffre d'affaires de 2021 des lignes que le levier touche (périodes de promotion du canal).

In [ ]:
# ---------------------------------------------------------------------------
# 0. Ce que rapporte un dollar de vente, et ce que rapporte un dollar de remise en moins (même calcul que 06)
# ---------------------------------------------------------------------------
commun_ex = commun[commun.annee == EXERCICE]
reel_ex = reel[reel.annee == EXERCICE]
COMPTES_VARIABLES = ["Autres charges"]          # seule charge qui suit les ventes avant impôts (06)
contribution = pd.DataFrame({"chiffre d'affaires net": commun_ex.groupby("type").ca_net.sum(),
                             "marge nette": commun_ex.groupby("type").marge_nette.sum(),
                             "charges qui suivent les ventes": reel_ex[reel_ex.nom_compte.isin(COMPTES_VARIABLES)].groupby("type").montant.sum(),
                             "résultat d'exploitation": commun_ex.groupby("type").resultat_exploitation.sum()}).loc[ORDRE_TYPES]
contribution.loc["Ensemble"] = contribution.sum()
contribution["ce que rapporte 1 $ de vente"] = (contribution["marge nette"] - contribution["charges qui suivent les ventes"]) / contribution["chiffre d'affaires net"]
contribution["ce que rapporte 1 $ de remise en moins"] = 1 - contribution["charges qui suivent les ventes"] / contribution["chiffre d'affaires net"]
display(contribution.style.format({c: ("{:.3f}" if "1 $" in c else "{:,.0f}") for c in contribution.columns}))
TAUX_CONTRIBUTION = float(contribution.loc["Ensemble", "ce que rapporte 1 $ de vente"])
TAUX_CONTRIBUTION_MAGASINS = float(contribution.loc["Magasins", "ce que rapporte 1 $ de vente"])
CONTRIBUTION_SEUIL = {"Magasins": TAUX_CONTRIBUTION_MAGASINS, "En ligne": TAUX_CONTRIBUTION, "Revendeur": TAUX_CONTRIBUTION, "Ensemble": TAUX_CONTRIBUTION}
print(f"Un dollar de vente rapporte {TAUX_CONTRIBUTION:.3f} $ de résultat (magasins : {TAUX_CONTRIBUTION_MAGASINS:.3f} $), comme dans 06".replace(".", ","))
tableaux["8_contribution"] = contribution

# ---------------------------------------------------------------------------
# 1. Les montants de 2021 et les écarts de marge mesurés (2019 à 2021)
# ---------------------------------------------------------------------------
ventes_ex = ventes[ventes.annee == EXERCICE]


def apport(marge_remisees, gain):
    """Marge que des lignes remisées apportent en plus de lignes non remisées équivalentes : marge × gain ÷ (1 + gain)."""
    return marge_remisees * gain / (1 + gain)


leviers = []
for t in ORDRE_TYPES:
    lt = ventes_ex[ventes_ex.type == t]
    en_promo = lt[lt.taux_remise_annonce > 0]
    k = contribution.loc[t, "ce que rapporte 1 $ de remise en moins"]
    g = {n: croise.loc[(t, NOMS_NIVEAUX[n]), "gain de marge (moyennes)"] for n in NIVEAUX}
    pt = par_type.loc[t]
    c_nouv = couples_type[t][couples_type[t].age == NOUVEAUTE]
    g_nouv, g_nouv_bas, g_nouv_haut = ecart_moyennes(c_nouv.marge_avec, c_nouv.marge_sans)
    marge_rem = lambda sel: lt[sel & lt.a_remise].marge_nette.sum()
    r20, r10 = lt[lt.taux_remise_annonce == 0.20].montant_remise.sum(), lt[lt.taux_remise_annonce == 0.10].montant_remise.sum()
    m20, m10 = marge_rem(lt.taux_remise_annonce == 0.20), marge_rem(lt.taux_remise_annonce == 0.10)
    nouv = lt.age == NOUVEAUTE
    definitions = [
        ("Supprimer les remises", lt.montant_remise.sum() * k,
         -apport(marge_rem(lt.taux_remise_annonce > 0), pt["gain de marge (moyennes)"]),
         -apport(marge_rem(lt.taux_remise_annonce > 0), pt["gain de marge, borne haute"]), -apport(marge_rem(lt.taux_remise_annonce > 0), pt["gain de marge, borne basse"]),
         en_promo.ca_net.sum()),
        ("Plafonner à 10 % (le 20 % ramené à 10 %)", r20 * (1 - 0.10 / 0.20) * k,
         m20 / (1 + g[0.20]) * (g[0.10] - g[0.20]), np.nan, np.nan, lt[lt.taux_remise_annonce == 0.20].ca_net.sum()),
        ("Plafonner à 7 % (le 10 % et le 20 % ramenés à 7 %)", (r20 * (1 - 0.07 / 0.20) + r10 * (1 - 0.07 / 0.10)) * k,
         m20 / (1 + g[0.20]) * (g[0.07] - g[0.20]) + m10 / (1 + g[0.10]) * (g[0.07] - g[0.10]), np.nan, np.nan,
         lt[lt.taux_remise_annonce.isin([0.10, 0.20])].ca_net.sum()),
        ("Cibler : plus de remise sur les nouveautés de l'année", lt[nouv].montant_remise.sum() * k,
         -apport(marge_rem(nouv), g_nouv), -apport(marge_rem(nouv), g_nouv_haut), -apport(marge_rem(nouv), g_nouv_bas),
         en_promo[en_promo.age == NOUVEAUTE].ca_net.sum()),
    ]
    for nom, gain_a, gain_b, b_bas, b_haut, exposees in definitions:
        leviers.append({"levier": nom, "canal": t, "gain lecture A": gain_a, "gain lecture B": gain_b,
                        "lecture B, borne basse": b_bas, "lecture B, borne haute": b_haut, "ventes exposées": exposees,
                        "ventes du canal": lt.ca_net.sum(), "résultat d'exploitation du canal": contribution.loc[t, "résultat d'exploitation"]})
leviers = pd.DataFrame(leviers)
total = leviers.groupby("levier", sort=False)[["gain lecture A", "gain lecture B", "lecture B, borne basse", "lecture B, borne haute", "ventes exposées", "ventes du canal",
                                               "résultat d'exploitation du canal"]].sum(min_count=1).reset_index().assign(canal="Ensemble")
leviers = pd.concat([leviers, total], ignore_index=True)
leviers["seuil : baisse des ventes exposées qui annule le gain A"] = leviers["gain lecture A"] / (leviers["ventes exposées"] * leviers.canal.map(CONTRIBUTION_SEUIL))
leviers["même seuil, en part des ventes du canal"] = leviers["gain lecture A"] / (leviers["ventes du canal"] * leviers.canal.map(CONTRIBUTION_SEUIL))
leviers["résultat du canal après (lecture A)"] = leviers["résultat d'exploitation du canal"] + leviers["gain lecture A"]
leviers["les deux lectures d'accord ?"] = np.where(np.sign(leviers["gain lecture A"]) == np.sign(leviers["gain lecture B"]), "oui : même signe", "non : signes opposés")
leviers = leviers.set_index(["levier", "canal"])
COLONNES_LEVIERS = ["gain lecture A", "gain lecture B", "lecture B, borne basse", "lecture B, borne haute", "les deux lectures d'accord ?",
                    "ventes exposées", "seuil : baisse des ventes exposées qui annule le gain A", "même seuil, en part des ventes du canal",
                    "résultat d'exploitation du canal", "résultat du canal après (lecture A)"]
print(f"Les leviers sur les remises, chiffrés sur l'exercice {EXERCICE} (gain de résultat d'exploitation par an) :")
display(leviers[COLONNES_LEVIERS].style.format({c: "{:.1%}" for c in COLONNES_LEVIERS if "seuil" in c}
                                               | {c: "{:,.0f}" for c in COLONNES_LEVIERS if "seuil" not in c and "accord" not in c}, na_rep=""))
tableaux["8_leviers"] = leviers

# ---------------------------------------------------------------------------
# 2. Le levier repris dans 06 : « supprimer les remises en magasin »
# ---------------------------------------------------------------------------
remises_magasins = ventes_ex[ventes_ex.type == "Magasins"].montant_remise.sum()
levier_06 = leviers.loc[("Supprimer les remises", "Magasins")]
recalcul = pd.Series({
    "remises des magasins 2021 (montant brut)": remises_magasins,
    "gain lecture A : remises × (1 − autres charges ÷ CA), formule de 06": levier_06["gain lecture A"],
    "lecture B (écart de marge mesuré, 2019 à 2021)": levier_06["gain lecture B"],
    "lecture B, borne basse": levier_06["lecture B, borne basse"], "lecture B, borne haute": levier_06["lecture B, borne haute"],
    "ventes exposées : CA des magasins en période de promotion": levier_06["ventes exposées"],
    "seuil : baisse de ces ventes qui annule le gain": levier_06["seuil : baisse des ventes exposées qui annule le gain A"],
    "même seuil, en part de toutes les ventes des magasins": levier_06["même seuil, en part des ventes du canal"],
    "résultat des magasins 2021": levier_06["résultat d'exploitation du canal"],
    "résultat des magasins après le levier": levier_06["résultat du canal après (lecture A)"],
}).to_frame(f"exercice {EXERCICE}")
print("Le levier « supprimer les remises en magasin » de 06, recalculé ici :")
display(recalcul.style.format(lambda v: pct(v, signe=False) if abs(v) < 1 else f"{v:,.0f}".replace(",", " ")))
print("Justification de 06, vérifiée par exercice (magasins, unités en plus dans une ligne remisée) :",
      {a: pct(verification.loc[("Magasins", a), "unités en plus (moyennes)"]) for a in ANNEES})
tableaux["8_levier_repris_dans_06"] = recalcul

# Second levier repris dans 06 : plafonner la remise à 10 % en ligne et chez le revendeur
#   lecture A = remises des lignes à 20 % × 0,5 × ce que rapporte 1 $ de remise en moins
plafond_06 = leviers.loc[[("Plafonner à 10 % (le 20 % ramené à 10 %)", t) for t in ["En ligne", "Revendeur"]]]
for t in ["En ligne", "Revendeur"]:
    controle = (ventes_ex[(ventes_ex.type == t) & (ventes_ex.taux_remise_annonce == 0.20)].montant_remise.sum() * 0.5
                * contribution.loc[t, "ce que rapporte 1 $ de remise en moins"])
    assert np.isclose(controle, leviers.loc[("Plafonner à 10 % (le 20 % ramené à 10 %)", t), "gain lecture A"]), "Plafond à 10 % : formule inattendue"
    print(f"Plafonner à 10 %, {t.lower()} : lecture A {m_dollars(controle, 2)} · lecture B {m_dollars(leviers.loc[('Plafonner à 10 % (le 20 % ramené à 10 %)', t), 'gain lecture B'], 2)}")
print(f"Plafonner à 10 % en ligne et chez le revendeur (levier repris dans 06) : lecture A {m_dollars(plafond_06['gain lecture A'].sum(), 2)}"
      f" · lecture B {m_dollars(plafond_06['gain lecture B'].sum(), 2)}")

# ---------------------------------------------------------------------------
# 3. Figure : la fourchette de chaque levier, canal par canal
# ---------------------------------------------------------------------------
trace = leviers.drop(index="Ensemble", level="canal")
fig, ax = plt.subplots(figsize=(11, 7))
y = np.arange(len(trace))[::-1]
for position, ((nom, t), b) in zip(y, trace.iterrows()):
    a_, b_ = b["gain lecture A"], b["gain lecture B"]
    ax.plot([min(a_, b_), max(a_, b_)], [position, position], color=viz.GRILLE, linewidth=9, solid_capstyle="butt", zorder=1)
    ax.scatter(a_, position, s=90, color=viz.PRINCIPALE, zorder=3, label="Lecture A : les lignes resteraient les mêmes" if position == y[0] else None)
    ax.scatter(b_, position, s=90, color=viz.ACCENT, zorder=3, label="Lecture B : la ligne ressemble à une ligne non remisée" if position == y[0] else None)
    droite, gauche = max(a_, b_), min(a_, b_)
    ax.text(droite + 0.4e6, position, m_dollars(droite, signe=True).replace("-", "−"), va="center", fontsize=8.5, color=viz.FONCE)
    ax.text(gauche - 0.4e6, position, m_dollars(gauche, signe=True).replace("-", "−"), va="center", ha="right", fontsize=8.5, color=viz.FONCE)
ax.axvline(0, color=viz.GRIS, linewidth=1)
COURTS = {"Supprimer les remises": "Supprimer", "Plafonner à 10 % (le 20 % ramené à 10 %)": "Plafonner à 10 %",
          "Plafonner à 7 % (le 10 % et le 20 % ramenés à 7 %)": "Plafonner à 7 %", "Cibler : plus de remise sur les nouveautés de l'année": "Cibler : rien sur les nouveautés"}
ax.set_yticks(y, [f"{t} · {COURTS[nom]}" for nom, t in trace.index], fontsize=9)
etendue = trace[["gain lecture A", "gain lecture B"]].values
ax.set_xlim(etendue.min() - 0.25 * np.ptp(etendue), etendue.max() + 0.25 * np.ptp(etendue))
viz.format_dollars(ax, axe="x")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2)
viz.titre(ax, "En magasin, réduire les remises rapporte dans les deux lectures ; ailleurs, les supprimer peut coûter",
          f"Gain de résultat d'exploitation par an selon la lecture, chiffré sur l'exercice {EXERCICE}")
print("Figure enregistrée :", viz.enregistrer(fig, "09_leviers_remises"))
plt.show()

**Lecture**

**Ce que rapporte un dollar.** Recalculé comme dans `06`, un dollar de vente rapporte 0,552 $ de résultat (0,551 $ en magasin). Un dollar de remise en moins, à ventes inchangées, rapporte 0,989 $ : il s'ajoute au chiffre d'affaires sans coût d'achat, moins les « autres charges » qui suivent les ventes (1,1 %). Un retour étant remboursé au prix catalogue, supprimer une remise ne change rien au montant des retours.

**Les leviers, chiffrés sur 2021** (gain de résultat d'exploitation par an) :

| Canal | Levier | Lecture A | Lecture B (intervalle) | Même signe ? | Baisse des ventes qui annule le gain : ventes exposées · tout le canal |
|---|---|---|---|---|---|
| Magasins | Supprimer | +5,4 M$ | +6,1 M$ (+5,9 à +6,3) | oui | 5,2 % · 3,7 % |
| Magasins | Plafonner à 10 % | +1,9 M$ | +2,0 M$ | oui | 4,6 % · 1,3 % |
| Magasins | Plafonner à 7 % | +2,8 M$ | +2,8 M$ | oui | 4,2 % · 1,9 % |
| Magasins | Cibler : rien sur les nouveautés | +0,7 M$ | +0,7 M$ | oui | 6,5 % · 0,4 % |
| En ligne | Supprimer | +5,2 M$ | −18,6 M$ (−19,4 à −17,6) | **non** | 5,0 % · 3,5 % |
| En ligne | Plafonner à 10 % | +1,8 M$ | +4,6 M$ | oui | 4,4 % · 1,2 % |
| En ligne | Plafonner à 7 % | +2,7 M$ | +0,6 M$ | oui | 4,0 % · 1,8 % |
| En ligne | Cibler : rien sur les nouveautés | +0,7 M$ | −2,6 M$ | **non** | 5,6 % · 0,5 % |
| Revendeur | Supprimer | +3,1 M$ | −7,4 M$ (−7,9 à −6,8) | **non** | 5,1 % · 3,6 % |
| Revendeur | Plafonner à 10 % | +1,1 M$ | +1,2 M$ | oui | 4,5 % · 1,3 % |
| Revendeur | Plafonner à 7 % | +1,6 M$ | +2,1 M$ | oui | 4,1 % · 1,9 % |
| Revendeur | Cibler : rien sur les nouveautés | +0,4 M$ | −1,1 M$ | **non** | 6,2 % · 0,5 % |

**Comment lire ce tableau.**

- **Magasins : supprimer les remises rapporte dans les deux lectures, 5,4 à 6,1 M$ par an.** La lecture B est même un peu plus haute que la lecture A, parce qu'une ligne remisée en magasin contient légèrement moins d'unités (section 5). Le déficit des magasins passerait de 10,8 à 5,4 M$. Le gain ne disparaîtrait que si les ventes des magasins en période de promotion reculaient de 5,2 % (3,7 % de toutes leurs ventes) du seul fait de l'absence de remise.
- **En ligne et chez le revendeur : ne pas supprimer sans essai.** Le gain va de +5,2 à −18,6 M$ en ligne et de +3,1 à −7,4 M$ chez le revendeur selon que les grosses lignes remisées resteraient ou partiraient avec la remise. L'écart est trop grand pour décider sur les données.
- **Plafonner donne le même signe dans les deux lectures, dans les trois canaux.** Ramener le 20 % à 10 % rapporte 4,9 M$ (A) à 7,8 M$ (B) sur l'ensemble ; tout ramener à 7 %, 7,1 M$ (A) à 5,6 M$ (B). C'est la conséquence de la section 6 : le niveau à 20 % ne fait pas de plus grosses lignes que les autres. Mais la lecture B compare des niveaux qui sont aussi des saisons : c'est un **indice**, à confirmer par un essai, pas un gain acquis.
- **Cibler par produit rapporte peu** (1,8 M$ en lecture A sur l'ensemble) et, hors magasins, les deux lectures s'opposent. La section 7 n'a trouvé aucune famille de produits où la remise serait plus ou moins utile : le ciblage qui a un sens est **par canal**, ce qui revient à supprimer les remises en magasin.
- **Le combiné défendable** : supprimer les remises en magasin (5,4 à 6,1 M$) et essayer un plafond à 10 % en ligne et chez le revendeur (2,9 M$ en lecture A, 5,8 M$ en lecture B), soit **8,4 à 11,9 M$ par an**.

**Les deux leviers repris dans `06`.**

| | Supprimer les remises en magasin | Plafonner la remise à 10 % en ligne et chez le revendeur |
|---|---|---|
| Formule (lecture A) | `montant_remise` des magasins 2021 × (1 − autres charges ÷ CA), soit × 0,989 | `montant_remise` des lignes à 20 % en 2021 × 0,5 × 0,989 |
| Gain lecture A | **5,42 M$** (remises brutes : 5,48 M$) | **2,93 M$** (en ligne 1,82 M$ + revendeur 1,10 M$) |
| Gain lecture B | 6,07 M$ (intervalle 5,85 à 6,29 M$) | 5,81 M$ (en ligne 4,58 M$ + revendeur 1,23 M$), à lire comme un indice |
| Justification | « en magasin, une ligne remisée ne contient pas plus d'unités qu'une autre », **vérifiée sur chaque exercice** (+0,7 %, −1,5 %, −1,9 % d'unités ; 98 % à 113 % de substitution), et vraie en magasin seulement : en ligne et chez le revendeur, les lignes remisées contiennent 19 % à 45 % d'unités en plus | le niveau à 20 % ne fait pas de plus grosses lignes que 7 % ou 10 % (section 6) ; niveau et saison restent confondus |
| Seuil (baisse des ventes qui annule le gain A) | 5,2 % des ventes des magasins en période de promotion (187,7 M$), soit 3,7 % de toutes leurs ventes, à 0,551 $ par dollar | 4,4 % (en ligne) et 4,5 % (revendeur) des ventes à 20 %, à 0,552 $ par dollar |

- **Cohérence avec `06`** : `06` calcule désormais ce levier avec la même formule nette des « autres charges » (1,1 % du chiffre d'affaires) ; les deux notebooks donnent donc la même valeur, 5,42 M$.
- **Ce que le constat ne couvre pas** : il porte sur l'intérieur d'une ligne. Il ne dit pas si, sans remise, des clients viendraient moins souvent (moins de lignes), question qui relève de `07_q2_promotions`. D'où le seuil, à associer à ce levier comme pour la publicité.

## 9. Le registre des tests

Tous les tests du notebook, avec le seuil corrigé (5 % divisé par le nombre de tests).

In [ ]:
verdict = tableau_tests(registre, ALPHA)
print(f"{len(verdict)} tests · seuil corrigé de Bonferroni : {ALPHA / len(verdict):.4f}".replace(".", ","))
pd.set_option("display.max_colwidth", 160)
display(verdict.style.format({"n": "{:,.0f}", "p": lambda p: "< 1e-300" if p == 0 else f"{p:.2g}", "seuil corrigé": "{:.4f}"}).hide(axis="index"))
tableaux["9_registre_des_tests"] = verdict

**Lecture du registre**

20 tests, seuil corrigé de 0,0025. Deux tests conservent H0, dix-huit la rejettent.

| Question | Résultat | Solidité |
|---|---|---|
| Un magasin qui remise plus de lignes vend-il plus, à magasin et mois constants ? | Non : ρ = −0,03 avec les unités, −0,04 avec la marge | H0 conservée ; le test manque de variation pour exclure un effet modéré |
| En magasin, une ligne remisée contient-elle plus d'unités ? (un test par exercice) | +0,7 %, −1,5 %, −1,9 % | H0 rejetée, mais l'effet est négligeable et change de signe : **pas plus d'unités** (borne haute de l'intervalle au plus +1,0 %) |
| En ligne et chez le revendeur ? (un test par exercice) | +19 % à +45 % | H0 rejetée, effet net et de même sens chaque exercice |
| En magasin, une ligne remisée rapporte-t-elle plus de marge ? | Non : −7,8 %, moyenne et médiane d'accord | H0 rejetée, effet net |
| En ligne et chez le revendeur ? | +29,6 % et +19,4 % en moyenne ; couple médian +23,2 % et −1,1 % | H0 rejetée ; chez le revendeur le sens dépend de la mesure retenue |
| Le niveau change-t-il le gain de marge ? (trois canaux) | Oui, et le 20 % est toujours le moins bon | H0 rejetée dans les trois canaux, à produit égal (pas à saison égale) |

**Remarque de méthode.** Les tests sur les unités en magasin rejettent H0 avec des écarts de 1 à 2 %, dans un sens puis dans l'autre. C'est l'illustration de la règle de lecture : avec des milliers de couples, **une valeur p ne remplace jamais l'examen de la taille de l'effet et de son intervalle**.

## 10. Synthèse : réponse à la question Q5

**Question Q5** : *quels niveaux de remise maximisent la marge ?*

### 1. Réponse

| Canal | Niveau qui maximise la marge | Appui |
|---|---|---|
| **Magasins** | **Aucune remise.** Chaque niveau fait perdre de la marge à une ligne, et la perte grandit avec le taux (−4 %, −6 %, −11 %) | Une ligne remisée contient autant d'unités qu'une autre, chaque exercice ; vrai dans toutes les gammes, tous les âges, toutes les catégories |
| **En ligne** | **10 %, en tout cas pas 20 %.** La ligne remisée est plus grosse à tous les niveaux, sans que le niveau fasse grossir la ligne | La remise va avec de plus grosses lignes ; qu'elle en soit la cause reste ouvert |
| **Revendeur** | **7 %, en tout cas pas 20 %.** Même lecture qu'en ligne, avec un gain plus faible et un cas typique qui ne bouge pas | Idem |

Le niveau à 20 %, qui fait 70 % à 72 % du coût des remises chaque exercice, n'est le meilleur dans aucun canal, aucun exercice.

**Mais les remises ne sont pas le problème prioritaire de l'entreprise.** Elles coûtent environ 2 % du chiffre d'affaires chaque exercice, et celles des magasins ont **baissé** de 6,9 M$ entre 2019 et 2021 (12,4 → 5,5 M$) pendant que leur résultat perdait 132,4 M$ (121,6 → −10,8 M$) : elles n'expliquent pas la dégradation. Le levier qu'elles offrent (5,42 M$ en magasin) est réel, mais il pèse bien moins que le renouvellement de l'assortiment (46,1 M$), la hausse des frais généraux (+26,3 M$ pour l'entreprise en deux exercices) ou la perte des grands comptes en ligne (tableau de hiérarchie, partie 4).

### 2. Les chiffres clés sur les trois exercices

| | 2019 | 2020 | 2021 |
|---|---|---|---|
| Remises, ensemble | 22,1 M$ · 2,33 % du CA | 15,5 M$ · 2,05 % | 13,9 M$ · 2,02 % |
| Remises, magasins | 12,4 M$ | 6,4 M$ | 5,5 M$ |
| Taux effectif, magasins | 2,33 % | 2,03 % | 2,01 % |
| Unités en plus dans une ligne remisée, magasins | +0,7 % | −1,5 % | −1,9 % |
| Unités en plus, en ligne · revendeur | +44,7 % · +33,8 % | +37,0 % · +27,2 % | +31,9 % · +18,9 % |
| Marge d'une ligne remisée, magasins | −7,1 % | −8,4 % | −8,1 % |
| Part de la gamme économique : CA · remises | 10,6 % · 18,0 % | 14,2 % · 13,1 % | 21,6 % · 10,4 % |

### 3. Registre des constats

| Constat | Verdict |
|---|---|
| Le niveau de remise est attaché à une saison, la même sur les trois exercices | Retenu |
| Une remise porte sur 1 à 4 unités par ligne, quels que soient le niveau, le canal et l'exercice | Retenu |
| En magasin, une ligne remisée ne contient pas plus d'unités qu'une autre (2019, 2020 et 2021) | Retenu |
| En ligne et chez le revendeur, une ligne remisée ne contient pas plus d'unités qu'une autre | Non soutenu : +19 % à +45 % chaque exercice |
| Un magasin qui remise plus de lignes vend plus | Non soutenu |
| En magasin, la remise fait perdre de la marge, d'autant plus que le taux est élevé | Retenu |
| En ligne et chez le revendeur, la remise est la cause des lignes plus grosses | Ouvert |
| Le niveau à 20 % rapporte moins de marge qu'un niveau plus bas, dans tous les canaux | Retenu à produit égal ; indice seulement, car niveau et saison sont confondus |
| Les remises ont suivi la descente en gamme | Non soutenu : la part de la gamme économique dans les remises est passée de 18,0 % à 10,4 % |
| Le recul du taux effectif vient de lignes plus grosses, à nombre d'unités remisées constant | Retenu |
| Une gamme, un âge ou une catégorie de produit fait exception | Non soutenu |
| Supprimer les remises en magasin rapporte, quelle que soit la lecture | Retenu (5,4 à 6,1 M$), sous réserve de l'effet sur le nombre de lignes (`07_q2_promotions`) |
| Supprimer les remises en ligne et chez le revendeur rapporte | Ouvert : de +8,3 à −25,9 M$ selon la lecture |
| Plafonner la remise à 10 % ou 7 % rapporte dans tous les canaux | Indice (même signe dans les deux lectures, niveau et saison confondus) |
| Les remises expliquent la dégradation du résultat des magasins | Non soutenu : leur montant a baissé de 6,9 M$ de 2019 à 2021, à 2 % du CA environ chaque exercice, pendant que le résultat perdait 132,4 M$ |

### 4. Recommandations chiffrées sur l'exercice 2021

| # | Recommandation | Gain par an | Seuil | Statut |
|---|---|---|---|---|
| 1 | **Arrêter les remises dans les magasins** | 5,4 à 6,1 M$ | perdre 5,2 % des ventes en période de promotion annulerait le gain | Appuyé par les données ; c'est aussi le test : si les ventes tiennent, il est confirmé, sinon on le voit en quelques semaines |
| 2 | **En ligne et chez le revendeur, essayer un plafond à 10 %** (le 20 % ramené à 10 %) sur une partie des produits, avec un groupe témoin | 2,9 M$ (A) à 5,8 M$ (B) | 4,4 % à 4,5 % des ventes à 20 % | Piste à tester |
| 3 | **Ne pas supprimer les remises en ligne et chez le revendeur sans essai** | de +8,3 à −25,9 M$ | | Ce qu'on ne recommande pas |
| 4 | **Repenser la promotion plutôt que la reconduire** : ciblée par canal, limitée dans le temps, mesurée avec un témoin | non chiffrable | | Recommandation de conception, avec `07_q2_promotions` |

Ensemble, les recommandations 1 et 2 rapportent **8,4 à 11,9 M$ par an**, dont 5,4 à 6,1 M$ dans les magasins (la recommandation 2 ne touche que la boutique en ligne et le revendeur). Avec les deux leviers de `06` qui ne touchent pas aux ventes (frais généraux au montant de 2019, +9,2 M$ ; publicité des magasins −10 %, +6,2 M$), le résultat des magasins passerait de −10,8 M$ à environ +10,0 à +10,7 M$ (−10,8 + 9,2 + 6,2 + 5,4 à 6,1), soit **un bénéfice de l'ordre de 10 M$** (ordre de grandeur : les leviers ne s'additionnent pas exactement, et la suppression des remises reste soumise à son seuil).

**Le levier repris dans `06`** : « supprimer les remises en magasin » vaut **5,42 M$**, avec la formule `montant_remise × (1 − autres charges ÷ CA)`, la même que celle qu'utilise désormais `06` : les deux notebooks sont cohérents. La justification est vérifiée sur les trois exercices et limitée aux magasins, avec un seuil de 5,2 % des ventes des magasins en période de promotion. Le levier « plafonner la remise à 10 % en ligne et chez le revendeur » vaut **2,93 M$** en lecture A (1,82 M$ en ligne, 1,10 M$ chez le revendeur ; 5,81 M$ en lecture B), à confirmer par un essai.

**Hiérarchie des problèmes : les remises et les retours sont secondaires.**

| Problème | Coût ou enjeu (exercice 2021) | Source |
|---|---|---|
| Assortiment non renouvelé | 552, 429 puis 272 lancements ; les nouveautés de l'année portent 69 % de la chute des ventes des magasins ; revenir au rythme de lancement de 2019 : **46,1 M$** | `06b_q1_chute_des_ventes`, `06_q1_facteurs_rentabilite` |
| Hausse des frais généraux (décisions de gestion) | magasins +12,9 M$ en deux exercices (83,9 → 96,8 M$), entreprise +26,3 M$ (155,0 → 181,3 M$) ; personnel, administration et informatique au montant de 2019 : **9,2 M$** | `06_q1_facteurs_rentabilite` |
| Perte des grands comptes de la boutique en ligne | entreprises clientes historiques : 169,0 M$ de ventes en 2019, 86,6 M$ en 2020, **0,1 M$ en 2021** (table détaillée des ventes en ligne : montants qui ne s'additionnent pas avec les autres lignes de ce tableau) | `06b_q1_chute_des_ventes` |
| Remises | 13,9 M$ accordées en 2021 (2,02 % du CA), dont 5,5 M$ en magasin ; supprimer en magasin : **5,42 M$** ; plafonner à 10 % ailleurs : **2,93 M$** (lecture A) | ce notebook |
| Retours | réduire les retours de moitié : **2,62 M$** | `06_q1_facteurs_rentabilite`, `08_q3_q4_retours` |

Les deux leviers sur les remises en magasin et sur les retours réunis (5,42 + 2,62 = 8,0 M$) valent moins d'un cinquième du levier sur l'assortiment, et moins que la hausse des frais généraux de l'entreprise en deux exercices. Ils restent valables, et ce sont les seuls qui ne demandent ni investissement ni réorganisation, mais ils sont **secondaires** : ils ne suffisent pas à redresser l'entreprise et ne doivent pas détourner l'attention de l'assortiment, des frais généraux et des grands comptes en ligne.

### 5. Points à valider avec les gens du métier

- **La règle d'attribution d'une remise** : pourquoi une ligne reçoit une remise et une autre non, et pourquoi elle porte sur 1 à 4 unités. Toute la comparaison suppose que cette attribution ne dépend pas de l'acheteur.
- **Les grosses lignes remisées en ligne et chez le revendeur** : sont-elles des commandes de professionnels qui négocient une remise de volume ? Si oui, la remise n'en est pas la cause, et la lecture A est la bonne.
- **L'effet d'un arrêt des remises sur la fréquentation des magasins**, que ce notebook ne mesure pas (`07_q2_promotions`) : à suivre pendant les premières semaines.
- **Le calendrier propre à Contoso Cambridge** (promotions « North America ») : est-il voulu ?

### 6. Limites

- **Niveau et saison confondus** : chaque niveau tombe à la même période les trois exercices ; la comparaison des niveaux est faite à produit égal, pas à saison égale.
- **Pas de client ni de ticket** : une ligne est un cumul par produit, par boutique et par jour. On ne peut pas dire si une grosse ligne est une grosse commande ou plusieurs acheteurs.
- **La lecture B** suppose que les lignes non remisées du même produit, le même mois, sont un bon point de comparaison pour une ligne remisée.

## Enregistrement des tableaux

In [ ]:
# Tous les tableaux de ce notebook dans un classeur Excel, un onglet par tableau,
# pour les reprendre dans le rapport d'analyse (livrable L4) sans relancer le code.
chemin_excel = analyse.exporter_tableaux(tableaux, config.L4 / "q5_tableaux.xlsx")
print("Classeur enregistré :", chemin_excel)
print("Onglets :", list(tableaux))